# Paper 1 — Notebook 2 (v10): riesgo, ruteo, backtest, cotas, basamiento y análisis de robustez

**Entrada** (Drive, producidas por el Notebook 1 v6): `MyDrive/acanav_paper1/cells.csv` y `monthly_sparse.parquet`  
**Entrada adicional** (producida por el Notebook 1b, etapa N): `MyDrive/acanav_paper1/addon/`  
**Salida**: `MyDrive/acanav_paper1/results_v3/` (o `results_v3fast/`) + `results_v3.zip`, que se descarga solo.

| etapa | qué hace |
|---|---|
| A | Descriptivo: perfil de borde por tipo de AMP (con IC bootstrap y leave-one-site-out), mapa del área de estudio |
| B | Riesgo: LightGBM Tweedie por región, validación rolling-origin, SHAP |
| C | Ruteo: RB-SRP sobre bloques de 0,6°, greedy secuencial round-robin + **pasada de re-optimización** |
| D | Backtest fuera de muestra, tests (Wilcoxon + Holm, Friedman), IC bootstrap por bloques |
| E | **Cota superior LP** del óptimo mensual (gap de optimalidad certificado) |
| F | **Benchmark exacto** de la heurística de una salida (programación dinámica) |
| G | **Enumeración de los 336 basamientos** de la flota actual, frontera de Pareto y validación fuera de muestra |
| H | Sensibilidad a los supuestos (incluye radio, barrido, velocidad de búsqueda y tamaño del bloque) |
| I | Ancho del anillo y jurisdicción (aguas a menos de 200 mn de una costa vs alta mar) |
| K | Predictores alternativos, latencia de datos, métricas por bloque |
| L | Cotas LP ampliadas, **solución exacta** de instancias reducidas (enumeración de rutas + programa entero) y de salidas individuales sobre la lista de candidatos (ramificación y corte) |
| M | Indicador de cobertura, selección con origen rodante, destacamentos estacionales, S* con configuración completa, búsqueda más intensa |
| N | **Datos diarios por buque, SAR y detalle no-take** (requiere el Notebook 1b) |
| O | Costo de aleatorizar el plan |
| P | Verificaciones de datos (esfuerzo interior por arte, cambio de 2023) y horas de traslado |

**Uso:** Run all (CPU). La corrida completa tarda unas 4 horas. Cada etapa guarda un caché en Drive: si la sesión se corta, vuelve a correr todo y las etapas ya hechas se cargan en segundos. Las etapas I a P están protegidas: si una falla, el error queda anotado al final del resumen y el resto sigue. Al final se descarga `results_v3.zip`.

In [ ]:
!pip -q install lightgbm shap pyarrow shapely global-land-mask gurobipy 2>/dev/null
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json, time, math, os, itertools, warnings
warnings.filterwarnings('ignore')
import scipy, scipy.sparse as sp
from scipy import stats
from scipy.optimize import linprog
import lightgbm as lgb
HAS_LGB=True
try:
    import shap; HAS_SHAP=True
except Exception:
    HAS_SHAP=False; print('(no shap: gain importance only)')
print('numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__, '| lightgbm', lgb.__version__)

In [ ]:
IN = os.environ.get('P1_IN', '/content/drive/MyDrive/acanav_paper1')
if 'P1_IN' not in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')

cells = pd.read_csv(f'{IN}/cells.csv')
try:
    obs = pd.read_parquet(f'{IN}/monthly_sparse.parquet')
except Exception:
    obs = pd.read_csv(f'{IN}/monthly_sparse.csv')
print(f'cells : {len(cells):,} rows, {cells.mpa_id.nunique()} MPAs')
print(f'obs   : {len(obs):,} cell-months with data, years {obs.year.min()}-{obs.year.max()}')
assert cells.cell_id.is_monotonic_increasing and (cells.cell_id.values==np.arange(len(cells))).all(), 'cell_id must be 0..n-1'
for c in ['y_viol','y_inside','y_press','all_hours','n_vessels']: assert c in obs.columns, c

## Configuración
Los parámetros marcados `ASSUMED` son supuestos declarados; la etapa H los barre.

In [ ]:
FAST_MODE   = bool(int(os.environ.get('P1_FAST', '0')))   # True: test 2024-2025 and reduced experiments (~10 min)
YEARS       = list(range(2019, 2026))
TEST_YEARS  = [2024, 2025] if FAST_MODE else [2022, 2023, 2024, 2025]
SEL_YEAR    = 2021                       # basing is selected on this year only (strictly before the test years)
Y0          = YEARS[0]; NT = len(YEARS)*12
OUT         = f'{IN}/results_{"v3fast" if FAST_MODE else "v3"}'
os.makedirs(OUT, exist_ok=True)
SEED        = 0

# --- routing / detection ---
BLK_DEG     = 0.6      # patrol block (6 cells of 0.1 deg = 3 of 0.2 deg), about 60 km
PMIN        = 0.8      # detection probability targeted by one search of a block (random search)   ASSUMED
PRESENCE    = 1.0      # probability that the activity of the month is present during a search       ASSUMED
RESERVE     = 0.15     # fraction of endurance withheld as fuel reserve                              ASSUMED
CREW_H      = 8.0      # crew flight-duty limit per sortie (h); binds the P-3ACH, whose endurance is 12 h
R_SCALE     = 1.0      # multiplier of the operating radius (sensitivity)
SWEEP_SCALE = 1.0      # multiplier of the sweep width (sensitivity)
SEARCH_SPEED= 1.0      # search speed as a fraction of transit speed (sensitivity)
EEZ_KM      = 370.4    # 200 nautical miles
K_POOL      = 150 if FAST_MODE else 300   # candidate blocks per sortie
SORTIES_PER_AC_MONTH = 10                 # sorties per aircraft per month                           ASSUMED
WEEKS_PER_MONTH = 4.3
COVER_BONUS = 0.0      # optional planning bonus for MPAs not yet visited in the month (0 = off)
REOPT_PASSES= 1        # re-optimization passes after the sequential construction
SORTIE_SEEDS= 3        # seeded starts of the single-sortie heuristic (besides the plain construction)
SORTIE_SWAP = True     # drop-and-refill improvement of the best route
W_NT        = 5.0      # policy weight of activity inside no-take zones                              ASSUMED
N_RANDOM    = 2 if FAST_MODE else 3
N_BOOT      = 2000 if FAST_MODE else 10000
LP_LEVELS   = 6        # radial levels of the LP upper bound

BASES = pd.DataFrame([
    ('IQQ','Iquique',        -70.18,-20.53,'group',      5),
    ('TQM','Torquemada',     -71.48,-32.95,'home',      99),
    ('THN','Talcahuano',     -73.10,-36.72,'group',      5),
    ('PMC','Puerto Montt',   -73.09,-41.44,'group',      5),
    ('PUQ','Punta Arenas',   -70.85,-53.00,'group',      5),
    ('PWM','Puerto Williams',-67.62,-54.93,'detachment', 3),
    ('IPC','Mataveri',      -109.42,-27.16,'forward',    2),
], columns=['base_id','name','lon','lat','tier','sorties_wk']).set_index('base_id')
PLATFORMS = pd.DataFrame([
    ('P3',  'P-3ACH', 555, 12.0, 1850, 40),
    ('C295','C-295',  480,  8.0, 1100, 30),
], columns=['plat_id','name','v_kmh','T_h','R_km','W_km']).set_index('plat_id')
COMPAT = {'P3':['TQM','IPC','PUQ'], 'C295':['TQM','IQQ','THN','PMC','PUQ','PWM']}
FLEET  = {'P3':2, 'C295':3}

# basing scenarios: {base: {platform: n}}
SCENARIOS = {
 'S0': {'TQM':{'P3':2,'C295':3}},
 'S1': {'TQM':{'P3':2,'C295':2}, 'PUQ':{'C295':1}},
 'S2': {'TQM':{'P3':2,'C295':2}, 'PWM':{'C295':1}},
 'S3': {'TQM':{'P3':1,'C295':3}, 'IPC':{'P3':1}},
 'S4': {'TQM':{'P3':2,'C295':2}, 'IQQ':{'C295':1}},
 'S5': {'TQM':{'P3':1,'C295':1}, 'IPC':{'P3':1}, 'PWM':{'C295':1}, 'IQQ':{'C295':1}},
 'S6': {b:{p:1 for p in PLATFORMS.index if b in COMPAT[p]} for b in BASES.index},   # one aircraft of each compatible type per base (9 aircraft)
}
SC_LABEL = {'S0':'S0 reference','S1':'S1 C-295 to PUQ','S2':'S2 C-295 to PWM','S3':'S3 P-3 to IPC',
            'S4':'S4 C-295 to IQQ','S5':'S5 combined','S6':'S6 one per base'}
FULL_METHOD_SCEN = ['S0','S5']                    # all five methods here; risk/hist/oracle elsewhere
METHODS_FULL  = ['risk','hist','uniform','random','oracle']
METHODS_LIGHT = ['risk','hist','oracle']
M_LABEL = {'risk':'risk','hist':'historical','uniform':'uniform','random':'random','oracle':'oracle'}

print(f'mode={"FAST" if FAST_MODE else "FULL"} | test={TEST_YEARS} | out={OUT}')
for k,v in SCENARIOS.items():
    print(f'  {SC_LABEL[k]:18s} {sum(sum(d.values()) for d in v.values())} aircraft  {v}')

## Helpers

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    R=6371.0; p1,p2=np.radians(lat1),np.radians(lat2)
    a=np.sin((p2-p1)/2)**2+np.cos(p1)*np.cos(p2)*np.sin(np.radians(lon2-lon1)/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))

def geodesic_circle(lon0, lat0, r_km, n=181):
    """Points at great-circle distance r_km from (lon0, lat0)."""
    d=r_km/6371.0; b=np.linspace(0,2*np.pi,n); p0=np.radians(lat0); l0=np.radians(lon0)
    p=np.arcsin(np.sin(p0)*np.cos(d)+np.cos(p0)*np.sin(d)*np.cos(b))
    l=l0+np.arctan2(np.sin(b)*np.sin(d)*np.cos(p0), np.cos(d)-np.sin(p0)*np.sin(p))
    return np.degrees(l), np.degrees(p)

def wilcoxon_paired(a, b, alternative='greater'):
    """Paired Wilcoxon signed-rank test; returns (statistic, p)."""
    d=np.asarray(a,float)-np.asarray(b,float); d=d[d!=0]
    if len(d)==0: return 0.0, 1.0
    r=stats.wilcoxon(d, alternative=alternative); return float(r.statistic), float(r.pvalue)

def holm(p):
    """Holm step-down adjusted p-values (NaN entries are ignored)."""
    p=np.asarray(p,float); out=np.full_like(p,np.nan); ok=np.where(~np.isnan(p))[0]
    if len(ok)==0: return out
    o=ok[np.argsort(p[ok])]; m=len(o); adj=np.maximum.accumulate((m-np.arange(m))*p[o]); out[o]=np.minimum(adj,1.0); return out

def friedman_nemenyi(M, names):
    """M: blocks x methods (larger is better)."""
    n,k=M.shape
    R=np.apply_along_axis(lambda r: pd.Series(-r).rank().values,1,M).mean(0)
    chi2=12*n/(k*(k+1))*(np.sum(R**2)-k*(k+1)**2/4)
    q={2:1.960,3:2.343,4:2.569,5:2.728,6:2.850}.get(k,2.728)
    return dict(chi2=float(chi2),p=float(stats.chi2.sf(chi2,k-1)),avg_rank=dict(zip(names,np.round(R,3))),
                CD=float(q*math.sqrt(k*(k+1)/(6*n))),n=int(n))

def boot_ratio_ci(a, b, block=3, nboot=None, seed=0):
    """Moving-block bootstrap (blocks of `block` consecutive months) for 100*(mean(a)/mean(b)-1). Returns (lo, hi) at 95%."""
    a=np.asarray(a,float); b=np.asarray(b,float); n=len(a); nboot=nboot or N_BOOT
    rng=np.random.default_rng(seed); nb=int(np.ceil(n/block)); st=rng.integers(0,n-block+1,size=(nboot,nb))
    idx=(st[:,:,None]+np.arange(block)[None,None,:]).reshape(nboot,-1)[:,:n]
    r=100*(a[idx].mean(1)/np.maximum(b[idx].mean(1),1e-12)-1)
    return float(np.percentile(r,2.5)), float(np.percentile(r,97.5))

def gini(x):
    x=np.sort(np.asarray(x,float)); n=len(x)
    if n==0 or x.sum()==0: return 0.0
    return float(2*np.sum(np.arange(1,n+1)*x)/(n*x.sum())-(n+1)/n)

def tex_escape(s): return str(s).replace('&','\\&').replace('%','\\%').replace('_','\\_')
def to_latex(df, path, caption, label, fmt=None):
    fmt=fmt or {}; cols=list(df.columns); al='l'+'r'*(len(cols)-1)
    L=['\\begin{table}[t]','\\centering',f'\\caption{{{caption}}}\\label{{{label}}}','\\small',
       f'\\begin{{tabular}}{{{al}}}','\\toprule',' & '.join(tex_escape(c) for c in cols)+' \\\\','\\midrule']
    for _,r in df.iterrows():
        row=[]
        for c in cols:
            v=r[c]; f=fmt.get(c)
            if f and isinstance(v,(int,float,np.floating,np.integer)) and not pd.isna(v): row.append(f.format(v))
            else: row.append(tex_escape(v) if not (isinstance(v,float) and pd.isna(v)) else '--')
        L.append(' & '.join(row)+' \\\\')
    L+=['\\bottomrule','\\end{tabular}','\\end{table}']
    open(path,'w').write('\n'.join(L)); return path

plt.rcParams.update({'font.size':9,'axes.labelsize':9,'legend.fontsize':8,'xtick.labelsize':8,'ytick.labelsize':8,
                     'font.family':'DejaVu Sans','axes.linewidth':0.8,'pdf.fonttype':42,'savefig.dpi':300})
C_NT,C_MU,C_RISK,C_HIST,C_UNI,C_ORA='#B03A2E','#2E6FB0','#B03A2E','#2E6FB0','#7F8C8D','#1E8449'
def savefig(fig, stem):
    fig.savefig(f'{OUT}/{stem}.pdf', bbox_inches='tight'); fig.savefig(f'{OUT}/{stem}.png', dpi=200, bbox_inches='tight'); plt.close(fig)

# ---------------- map helpers (Mercator base map, Natural Earth 1:10m) ----------------
import urllib.request
import matplotlib.patheffects as pe
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Patch
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
MAP_EXT=(-127.0,-57.0,-62.5,-14.0)                      # lon west, lon east, lat south, lat north
C_OCEAN,C_LAND,C_COAST='#EAF1F6','#EFEBE1','#8E8E8E'
C_NTF,C_MUF,C_MPAE='#8FD0C4','#DCEFEA','#1F6F63'       # no-take fill, multiple-use fill, MPA edge
C_P3,C_C295='#1F5A99','#D9730D'
HALO=[pe.withStroke(linewidth=1.6,foreground='white')]
def merc(lat): return np.degrees(np.log(np.tan(np.pi/4+np.radians(np.asarray(lat,float))/2)))

def load_basemap(cache=None):
    """Natural Earth 1:10m land, minor islands and country borders clipped to the study area.
    Falls back to a gridded land mask, and to no land at all, if the files cannot be downloaded."""
    cache=cache or f'{IN}/naturalearth'; os.makedirs(cache,exist_ok=True)
    try:
        from shapely.geometry import shape, box
        clip=box(MAP_EXT[0]-8,MAP_EXT[2]-6,MAP_EXT[1]+8,MAP_EXT[3]+8); out=dict(kind='poly',land=[],borders=[])
        for name in ['ne_10m_land','ne_10m_minor_islands','ne_10m_admin_0_boundary_lines_land']:
            fn=f'{cache}/{name}.geojson'
            if not os.path.exists(fn):
                urllib.request.urlretrieve(f'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/{name}.geojson',fn)
            for ft in json.load(open(fn))['features']:
                g=shape(ft['geometry'])
                if not g.intersects(clip): continue
                g=g.intersection(clip)
                for p in getattr(g,'geoms',[g]):
                    if p.geom_type=='Polygon': out['land'].append([np.asarray(p.exterior.coords)]+[np.asarray(r.coords) for r in p.interiors])
                    elif p.geom_type=='LineString': out['borders'].append(np.asarray(p.coords))
        return out
    except Exception as e:
        print('Natural Earth not available:',e)
    try:
        from global_land_mask import globe
        la=np.arange(-66,-8,0.04); lo=np.arange(-136,-48,0.04); LO,LA=np.meshgrid(lo,la)
        return dict(kind='mask',lon=lo,lat=la,mask=globe.is_land(LA,LO).astype(float))
    except Exception: return None
BASEMAP=None

def draw_base(ax,ext=MAP_EXT,lon_ticks=None,lat_ticks=None,label_left=True):
    """Ocean, land, country borders, graticule and degree labels on a Mercator axis (x = longitude, y = merc(latitude))."""
    ax.set_facecolor(C_OCEAN)
    if BASEMAP is not None and BASEMAP['kind']=='poly':
        for rings in BASEMAP['land']:
            verts=[]; codes=[]
            for r in rings:
                verts+=[(x,float(merc(y))) for x,y in r[:,:2]]; codes+=[Path.MOVETO]+[Path.LINETO]*(len(r)-2)+[Path.CLOSEPOLY]
            ax.add_patch(PathPatch(Path(verts,codes),facecolor=C_LAND,edgecolor=C_COAST,linewidth=0.3,zorder=1))
        for b in BASEMAP['borders']: ax.plot(b[:,0],merc(b[:,1]),color='#B5B5B5',lw=0.35,zorder=1.1)
    elif BASEMAP is not None:
        ax.contourf(BASEMAP['lon'],merc(BASEMAP['lat']),BASEMAP['mask'],levels=[0.5,1.5],colors=[C_LAND],zorder=1)
        ax.contour(BASEMAP['lon'],merc(BASEMAP['lat']),BASEMAP['mask'],levels=[0.5],colors=[C_COAST],linewidths=0.3,zorder=1)
    lon_ticks=lon_ticks if lon_ticks is not None else np.arange(-120,-59,10); lat_ticks=lat_ticks if lat_ticks is not None else np.arange(-60,-19,10)
    ax.set_xlim(ext[0],ext[1]); ax.set_ylim(float(merc(ext[2])),float(merc(ext[3]))); ax.set_aspect('equal')
    ax.set_xticks(lon_ticks); ax.set_xticklabels([f'{-x:.0f}°W' for x in lon_ticks])
    ax.set_yticks(merc(lat_ticks)); ax.set_yticklabels([f'{-y:.0f}°S' for y in lat_ticks] if label_left else [])
    ax.grid(color='white',lw=0.5,zorder=0.5); ax.set_axisbelow(True); ax.tick_params(length=2,labelsize=6.5,width=0.5)
    for s_ in ax.spines.values(): s_.set_linewidth(0.6)

def raster(lon,lat,delta,val=None,step=0.1,origin=(-128.4,-64.2),shape=(540,700)):
    """Rasterizes cells on a regular grid of `step` degrees; a cell coarser than the grid fills all the grid cells it covers.
    Returns (lon centers, merc(lat centers), lon edges, merc(lat edges), grid) with NaN where there is no cell."""
    ny,nx=shape; G=np.full((ny,nx),np.nan); lon=np.asarray(lon,float); lat=np.asarray(lat,float); delta=np.asarray(delta,float)
    val=np.ones(len(lon)) if val is None else np.asarray(val,float)
    k=np.maximum(np.round(delta/step).astype(int),1)
    for kk in np.unique(k):
        m=k==kk; offs=(np.arange(kk)-(kk-1)/2)*step
        for dx in offs:
            for dy in offs:
                ix=np.floor((lon[m]+dx-origin[0])/step+1e-6).astype(int); iy=np.floor((lat[m]+dy-origin[1])/step+1e-6).astype(int)
                ok=(ix>=0)&(ix<nx)&(iy>=0)&(iy<ny); G[iy[ok],ix[ok]]=val[m][ok]
    xe=origin[0]+step*np.arange(nx+1); ye=origin[1]+step*np.arange(ny+1)
    return xe[:-1]+step/2, merc(ye[:-1]+step/2), xe, merc(ye), G

def draw_mpas(ax,fill=True,lw=0.6,mu_ls='-'):
    """Protected sites from their interior cells: no-take and multiple-use areas, and markers for sites smaller than a few cells."""
    inter=cells[cells.zone=='interior']
    for typ,fc,ls in [('multi-use',C_MUF,mu_ls),('no-take',C_NTF,'-')]:
        d=inter[inter.mpa_type==typ]; xc,yc,_,_,G=raster(d.lon,d.lat,d.delta); Z=np.nan_to_num(G)
        if fill: ax.contourf(xc,yc,Z,levels=[0.5,1.5],colors=[fc],zorder=2,alpha=0.95)
        ax.contour(xc,yc,Z,levels=[0.5],colors=[C_MPAE],linewidths=lw,linestyles=[ls],zorder=2.2)
    small=[]
    for (name,typ),d in cells.groupby(['mpa_name','mpa_type']):
        di=d[d.zone=='interior']
        if (di.area_km2.sum()<2500):
            p=di if len(di) else d.reindex(d.dist_bnd_km.abs().sort_values().index[:5]); small.append((p.lon.mean(),p.lat.mean(),typ))
    for x,y,typ in small:
        ax.plot(x,merc(y),marker='D',ms=3.2,mfc=C_MPAE if typ=='no-take' else 'white',mec=C_MPAE,mew=0.6,ls='',zorder=6)

def draw_bases(ax,which=None,names=True,fs=6.5):
    lab={'IQQ':('Iquique',5,0,'left'),'TQM':('Torquemada',5,0,'left'),'THN':('Talcahuano',5,0,'left'),'PMC':('Puerto\nMontt',5,0,'left'),
         'PUQ':('Punta\nArenas',5,5,'left'),'PWM':('Puerto\nWilliams',5,-7,'left'),'IPC':('Mataveri',0,-8,'center')}
    for b,r in BASES.iterrows():
        if which is not None and b not in which: continue
        ax.plot(r.lon,merc(r.lat),marker='s',ms=3.6,color='k',mec='white',mew=0.4,ls='',zorder=9)
        t,dx,dy,ha=lab[b]
        ax.annotate(t if names else b,(r.lon,float(merc(r.lat))),xytext=(dx,dy),textcoords='offset points',ha=ha,va='center',fontsize=fs,fontweight='bold',zorder=10,path_effects=HALO,linespacing=0.9)

def panel_title(ax,txt):
    ax.text(0.012,0.988,txt,transform=ax.transAxes,va='top',ha='left',fontsize=7.5,fontweight='bold',zorder=20,
            bbox=dict(boxstyle='square,pad=0.3',fc='white',ec='#555555',lw=0.5))

import pickle, hashlib
def _sig(): return hashlib.md5(json.dumps(dict(P=PMIN,q=PRESENCE,e=RESERVE,H=CREW_H,K=K_POOL,n=SORTIES_PER_AC_MONTH,w=W_NT,cb=COVER_BONUS,ss=SORTIE_SEEDS,sw=SORTIE_SWAP,ro=REOPT_PASSES,ty=TEST_YEARS,pl=PLATFORMS.values.tolist(),ba=BASES.sorties_wk.tolist(),sc={k_:SCENARIOS[k_] for k_ in sorted(SCENARIOS) if k_ in ('S0','S1','S2','S3','S4','S5','S6')},nr=N_RANDOM),sort_keys=True,default=str).encode()).hexdigest()[:8]
def cache_load(name):
    fn=f'{OUT}/cache_{name}_{_sig()}.pkl'
    if os.path.exists(fn): print(f'  (cache: {name})'); return pickle.load(open(fn,'rb'))
    return None
def cache_save(name,obj): pickle.dump(obj,open(f'{OUT}/cache_{name}_{_sig()}.pkl','wb'))
RESULTS={}
print('helpers ready')

## A. Resultados descriptivos (sin modelo)

In [ ]:
_o=cells[cells.zone=='interior'].assign(o=lambda d: d.no_take_kind.map({'none':0,'part':1,'all':2}).fillna(0))
mpa_kind=_o.groupby('mpa_id').o.max().map({0:'multi-use',1:'partial',2:'no-take'})
cells['mpa_type']=cells.mpa_id.map(mpa_kind).fillna('multi-use')
obs['t']=(obs.year-Y0)*12+obs.month-1
oc=obs.merge(cells[['cell_id','mpa_id','mpa_name','region','mpa_type','zone','dist_bnd_km','sector','reachable']], on='cell_id')
cells['h_total']=obs.groupby('cell_id').all_hours.sum().reindex(cells.cell_id).fillna(0).values

# ---- T1: table by MPA ----
t1=(oc.groupby('mpa_name').agg(region=('region',lambda s_: s_.mode().iloc[0]), mpa_type=('mpa_type','first'),
                               y_nt=('y_viol','sum'), y_inside=('y_inside','sum'), y_press=('y_press','sum')).reset_index())
ncell=cells.groupby('mpa_name').agg(cells=('cell_id','size'), interior=('zone',lambda s:(s=='interior').sum())).reset_index()
t1=t1.merge(ncell,on='mpa_name').sort_values('y_press',ascending=False)
t1['press_share_%']=100*t1.y_press/t1.y_press.sum()
to_latex(t1[['mpa_name','region','mpa_type','cells','interior','y_nt','y_inside','y_press','press_share_%']]
         .rename(columns={'mpa_name':'MPA','mpa_type':'type','press_share_%':'share (%)'}),
         f'{OUT}/T1_mpas.tex','Marine protected areas in the study, 2019--2025 (hours of apparent fishing effort).','tab:mpas',
         fmt={'y_nt':'{:,.0f}','y_inside':'{:,.0f}','y_press':'{:,.0f}','share (%)':'{:.1f}','cells':'{:,.0f}','interior':'{:,.0f}'})
print(t1[['mpa_name','mpa_type','y_nt','y_press','press_share_%']].to_string(index=False))

# ---- boundary profile by MPA type (hours per cell; denominator = all cells of the band) ----
bins=[-1e9,-100,-50,-20,0,20,50,100,200,400,800,1e9]
lab=['<-100','-100/-50','-50/-20','-20/0','0/20','20/50','50/100','100/200','200/400','400/800','>800']
cells['band']=pd.cut(cells.dist_bnd_km,bins=bins,labels=lab)
prof={}; rng=np.random.default_rng(SEED); BND={}
for typ in ['no-take','multi-use']:
    c=cells[cells.mpa_type==typ]
    p=c.groupby('band',observed=False).agg(h=('h_total','sum'),cells=('cell_id','size')); p['h_per_cell']=p.h/p.cells.replace(0,np.nan); prof[typ]=p
    hi=c[c.band=='-20/0'].h_total.values; ho=c[c.band=='0/20'].h_total.values; ratio=ho.mean()/hi.mean()
    bs=np.array([rng.choice(ho,len(ho)).mean()/max(rng.choice(hi,len(hi)).mean(),1e-9) for _ in range(N_BOOT)])
    loo=[]
    for m in c.mpa_id.unique():
        cc=c[c.mpa_id!=m]; a=cc[cc.band=='-20/0'].h_total; b=cc[cc.band=='0/20'].h_total
        if len(a) and len(b) and a.mean()>0: loo.append(b.mean()/a.mean())
    BND[typ]=dict(inside=float(hi.mean()),outside=float(ho.mean()),ratio=float(ratio),ci_lo=float(np.percentile(bs,2.5)),ci_hi=float(np.percentile(bs,97.5)),
                  loo_min=float(min(loo)),loo_max=float(max(loo)),n_in=int(len(hi)),n_out=int(len(ho)),sites=int(c.mpa_id.nunique()))
    print(f'\n[{typ}] boundary ratio {ratio:.1f}x  (95% CI {BND[typ]["ci_lo"]:.1f}-{BND[typ]["ci_hi"]:.1f}; leave-one-site-out {min(loo):.1f}-{max(loo):.1f})')
    print(p.round(2).to_string())
RESULTS['boundary']=BND
# share of the 0-20 km interior band that carries any effort, and per-site table (supplement)
site=(cells[cells.band.isin(['-20/0','0/20'])].groupby(['mpa_name','mpa_type','region','band'],observed=True).agg(cells=('cell_id','size'),h=('h_total','sum')).reset_index())
site['h_per_cell']=site.h/site.cells; site.to_csv(f'{OUT}/boundary_by_site.csv',index=False)
rt=(cells.groupby(['region','mpa_type','band'],observed=False).h_total.sum()/cells.groupby(['region','mpa_type','band'],observed=False).size().replace(0,np.nan)).unstack('band').round(1)
print('\nhours per cell by region x type:'); print(rt.to_string()); rt.to_csv(f'{OUT}/boundary_by_region.csv')

# ---- F3: boundary profile ----
fig,ax=plt.subplots(figsize=(5.2,3.2))
keep=[l for l in lab if (prof['no-take'].cells.get(l,0)+prof['multi-use'].cells.get(l,0))>0]
x=np.arange(len(keep))
for typ,col,mk,ls in [('no-take',C_NT,'o','-'),('multi-use',C_MU,'s','--')]:
    y=prof[typ].h_per_cell.reindex(keep).values
    ax.plot(x,np.where(y>0,y,np.nan),marker=mk,ms=5,color=col,lw=1.6,ls=ls,label=f"{typ.replace('multi-use','multiple-use')} sites")
xb=keep.index('0/20')-0.5
ax.axvline(xb,color='k',ls=':',lw=1.0); ax.text(xb-0.12,0.05,'inside',transform=ax.get_xaxis_transform(),fontsize=8,ha='right'); ax.text(xb+0.12,0.05,'outside',transform=ax.get_xaxis_transform(),fontsize=8)
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels([k.replace('/',' to ').replace('<-100','< -100') for k in keep],rotation=40,ha='right')
ax.set_xlabel('Signed distance to site boundary (km)'); ax.set_ylabel('Apparent fishing effort\n(hours per cell, 2019–2025)')
ax.legend(frameon=False,loc='upper left'); ax.spines[['top','right']].set_visible(False); ax.grid(alpha=0.25,which='major')
savefig(fig,'F3_boundary_profile')

# ---- F2: study area (a) and apparent fishing effort (b) ----
BASEMAP=load_basemap()
plt.rcParams['hatch.linewidth']=0.3
fig,axs=plt.subplots(1,2,figsize=(7.1,3.75))
# (a) geography and reach
ax=axs[0]; draw_base(ax)
xc,yc,_,_,G=raster(cells.lon,cells.lat,cells.delta); ax.contour(xc,yc,np.nan_to_num(G),levels=[0.5],colors=['#5D6D7E'],linewidths=0.5,linestyles=':',zorder=1.8)
ur=cells[cells.reachable==0]
if len(ur):
    _,_,_,_,GU=raster(ur.lon,ur.lat,ur.delta); cs=ax.contourf(xc,yc,np.nan_to_num(GU),levels=[0.5,1.5],colors='none',hatches=['//////'],zorder=1.9)
    for c_ in getattr(cs,'collections',[cs]): c_.set_edgecolor('#7B8794'); c_.set_linewidth(0)
draw_mpas(ax)
for b,r in BASES.iterrows():
    plat='P3' if b in COMPAT['P3'] else 'C295'; lo,la=geodesic_circle(r.lon,r.lat,PLATFORMS.loc[plat,'R_km'])
    ax.plot(lo,merc(la),ls='-' if plat=='P3' else (0,(4,2)),lw=0.6,color=C_P3 if plat=='P3' else C_C295,alpha=0.9,zorder=3)
draw_bases(ax)
for t,x,y,ha in [('Rapa Nui',-112.6,-22.4,'center'),('Motu Motiro\nHiva',-103.3,-31.6,'center'),('Nazca-\nDesventuradas',-84.4,-27.6,'right'),('Juan\nFernández',-85.3,-36.3,'right'),
                 ('Diego Ramírez–\nPaso Drake',-75.2,-59.0,'right'),('Kawésqar',-77.6,-51.6,'right'),('Tortel',-77.6,-47.6,'right')]:
    ax.text(x,float(merc(y)),t,fontsize=6,style='italic',color='#174F47',ha=ha,va='center',zorder=8,path_effects=HALO,linespacing=0.95)
ax.text(-96,float(merc(-45)),'PACIFIC OCEAN',fontsize=6,color='#7F9BB3',ha='center',zorder=1.5)
panel_title(ax,'(a) Protected sites, bases, and reach')
hd=[Patch(fc=C_NTF,ec=C_MPAE,lw=0.6,label='No-take site'),Patch(fc=C_MUF,ec=C_MPAE,lw=0.6,label='Multiple-use site'),
    Line2D([],[],marker='D',ms=3.2,mfc='white',mec=C_MPAE,mew=0.6,ls='',label='Site smaller than 2,500 km²'),
    Line2D([],[],color='#5D6D7E',lw=0.6,ls=':',label='Limit of the exterior rings'),Patch(fc='none',ec='#7B8794',lw=0.3,hatch='//////',label='Beyond the reach of all aircraft'),
    Line2D([],[],marker='s',ms=3.6,color='k',ls='',label='Naval air base'),Line2D([],[],color=C_P3,lw=0.8,label='P-3ACH operating radius'),Line2D([],[],color=C_C295,lw=0.8,ls=(0,(4,2)),label='C-295 operating radius')]
ax.legend(handles=hd,loc='lower left',fontsize=5.6,frameon=True,framealpha=0.95,edgecolor='#888888',borderpad=0.5,labelspacing=0.3,handlelength=1.8).set_zorder(15)
# (b) effort density
ax=axs[1]; draw_base(ax,label_left=False)
ring=cells[(cells.zone=='ring')&(cells.h_total>0)]
_,_,xe,ye,GH=raster(ring.lon,ring.lat,ring.delta,val=1000*ring.h_total/ring.area_km2)
pm=ax.pcolormesh(xe,ye,np.ma.masked_invalid(GH),cmap='YlOrRd',norm=LogNorm(vmin=0.1,vmax=3000),zorder=1.6,rasterized=True)
ax.contour(xc,yc,np.nan_to_num(G),levels=[0.5],colors=['#5D6D7E'],linewidths=0.5,linestyles=':',zorder=1.8)
draw_mpas(ax); draw_bases(ax,names=False)
panel_title(ax,'(b) Apparent fishing effort in the rings, 2019–2025')
cax=ax.inset_axes([0.05,0.075,0.44,0.028]); cb=fig.colorbar(pm,cax=cax,orientation='horizontal'); cb.outline.set_linewidth(0.4)
cb.set_ticks([0.1,1,10,100,1000]); cb.set_ticklabels(['0.1','1','10','100','1,000']); cax.tick_params(labelsize=5.6,length=1.5,width=0.4)
cax.set_title('Hours per 1,000 km²',fontsize=5.8,pad=2)
plt.subplots_adjust(wspace=0.04,left=0.06,right=0.995,top=0.99,bottom=0.07)
fig.savefig(f'{OUT}/F2_map.pdf',bbox_inches='tight',dpi=400); fig.savefig(f'{OUT}/F2_map.png',dpi=200,bbox_inches='tight'); plt.close(fig); print('F2 saved')
# ---- distance to the nearest coast: waters within 200 nm of a coast (national jurisdiction) versus high seas ----
cells['d_coast_km']=np.nan
if BASEMAP is not None and BASEMAP['kind']=='poly':
    from scipy.spatial import cKDTree
    _p=np.vstack([r_[0][:,:2] for r_ in BASEMAP['land']]); _p=_p[(_p[:,1]>-60)&(_p[:,1]<MAP_EXT[3]+7.5)&(_p[:,0]>MAP_EXT[0]-7.5)&(_p[:,0]<MAP_EXT[1]+7.5)]     # drops the clip frame and the Antarctic Treaty area
    def _xyz(lo,la): lo,la=np.radians(lo),np.radians(la); return np.c_[np.cos(la)*np.cos(lo),np.cos(la)*np.sin(lo),np.sin(la)]
    _d,_=cKDTree(_xyz(_p[:,0],_p[:,1])).query(_xyz(cells.lon.values,cells.lat.values)); cells['d_coast_km']=2*6371.0*np.arcsin(_d/2)
cells['jur']=np.where(cells.d_coast_km.notna(),(cells.d_coast_km<=EEZ_KM).astype(float),np.nan)
_r=cells[cells.zone=='ring']; RESULTS['jurisdiction']=dict(available=bool(cells.jur.notna().all()),ring_pressure_share=float(_r[_r.jur==1].h_total.sum()/_r.h_total.sum()),ring_cells_share=float((_r.jur==1).mean()),
    by_site={k_:round(float(v_),4) for k_,v_ in (_r[_r.jur==1].groupby('mpa_name').h_total.sum()/_r.groupby('mpa_name').h_total.sum()).fillna(0).items()},
    by_ring_width={str(w_):float(_r[_r.dist_bnd_km<=w_].h_total.sum()/_r.h_total.sum()) for w_ in [50,100,200,400]})
print('ring pressure within 200 nm of a coast: %.1f%%'%(100*RESULTS['jurisdiction']['ring_pressure_share']),'| by ring width:',{k_:round(v_,3) for k_,v_ in RESULTS['jurisdiction']['by_ring_width'].items()})

# ---- composition by gear, year, sector; reachability ----
gear=obs[['h_trawl','h_longline','h_squid','h_seine','h_other']].sum(); RESULTS['gear_hours']=gear.round(0).to_dict()
yr=obs.groupby('year').agg(y_nt=('y_viol','sum'),y_press=('y_press','sum')); RESULTS['by_year']=yr.round(0).to_dict()
sec=oc.assign(sector=np.where(oc.reachable==1,oc.sector,'unreachable')).groupby('sector').agg(y_nt=('y_viol','sum'),y_press=('y_press','sum'))
sec['cells']=cells.assign(sector=np.where(cells.reachable==1,cells.sector,'unreachable')).groupby('sector').size(); sec.to_csv(f'{OUT}/sectors.csv'); print('\n',sec.round(1).to_string())
RESULTS.update(dict(n_cells=int(len(cells)), n_mpas=int(cells.mpa_id.nunique()), n_obs=int(len(obs)),
                    y_nt_total=float(obs.y_viol.sum()), y_press_total=float(obs.y_press.sum()), y_inside_total=float(obs.y_inside.sum()),
                    nt_top_mpa=t1.sort_values('y_nt',ascending=False).iloc[0].mpa_name, nt_top_share=float(t1.y_nt.max()/t1.y_nt.sum()),
                    press_top2_share=float(t1.y_press.nlargest(2).sum()/t1.y_press.sum()), unreachable_press=float(oc[oc.reachable==0].y_press.sum())))
_ur=oc[oc.reachable==0].groupby('year').all_hours.sum(); _tot=oc.groupby('year').all_hours.sum()
RESULTS['unreachable_share_by_year']={int(k):round(float(v),4) for k,v in (_ur.reindex(_tot.index).fillna(0)/_tot).items()}
RESULTS['unreachable_share_test']=float(_ur.reindex(TEST_YEARS).fillna(0).sum()/_tot.reindex(TEST_YEARS).sum())
print('unreachable share by year:',RESULTS['unreachable_share_by_year'])

## B.1 Panel denso y variables

In [ ]:
n=len(cells); t0=time.time()
def dense(col, dtype=np.float32):
    M=np.zeros((n,NT),dtype); np.add.at(M,(obs.cell_id.values,obs.t.values),obs[col].values.astype(dtype)); return M
M_all=dense('all_hours'); M_viol=dense('y_viol'); M_press=dense('y_press'); M_ves=dense('n_vessels')
M_gear={g:dense(f'h_{g}') for g in ['longline','squid','trawl','seine']}

def lag(M,k):  out=np.full_like(M,np.nan); out[:,k:]=M[:,:-k]; return out
def roll_mean(M,w):  # media de los w meses ANTERIORES (t-w..t-1); cs[:,t] = suma de M[:,:t]
    cs=np.cumsum(np.concatenate([np.zeros((n,1),M.dtype),M],1),1); out=np.full_like(M,np.nan)
    out[:,w:]=(cs[:,w:NT]-cs[:,0:NT-w])/w; return out
def roll_max(M,w):
    out=np.full_like(M,np.nan)
    for t in range(w,NT): out[:,t]=M[:,t-w:t].max(1)
    return out
def months_since(M):
    out=np.zeros_like(M); ms=np.full(n,36.0)
    for t in range(NT):
        out[:,t]=ms; ms=np.where(M[:,t]>0,0.0,np.minimum(ms+1,36.0))
    return out
def n_active(M,w):
    A=(M>0).astype(np.float32); cs=np.cumsum(np.concatenate([np.zeros((n,1),np.float32),A],1),1)
    out=np.full_like(M,np.nan); out[:,w:]=cs[:,w:NT]-cs[:,0:NT-w]; return out

# vecindad espacial (8 vecinos, misma resolucion) sobre lag1
key={}
for d in cells.delta.unique():
    sub=cells[cells.delta==d]; key[d]={(round(x,6),round(y,6)):i for x,y,i in zip(sub.lon,sub.lat,sub.index)}
nb_idx=np.full((n,8),-1)
for i,(x,y,d) in enumerate(zip(cells.lon,cells.lat,cells.delta)):
    j=0
    for dx in (-d,0,d):
        for dy in (-d,0,d):
            if dx==0 and dy==0: continue
            nb_idx[i,j]=key[d].get((round(x+dx,6),round(y+dy,6)),-1); j+=1
L1=lag(M_all,1); L1z=np.nan_to_num(L1)
nb_sum=np.zeros_like(M_all); nb_cnt=(nb_idx>=0).sum(1).astype(np.float32)
for j in range(8):
    ok=nb_idx[:,j]>=0; nb_sum[ok]+=L1z[nb_idx[ok,j]]
nb_mean=np.where(nb_cnt[:,None]>0, nb_sum/np.maximum(nb_cnt,1)[:,None], 0)

F={}
F['lag1']=L1; F['lag2']=lag(M_all,2); F['lag3']=lag(M_all,3); F['lag12']=lag(M_all,12)
F['roll3']=roll_mean(M_all,3); F['roll12']=roll_mean(M_all,12); F['max12']=roll_max(M_all,12)
F['nact12']=n_active(M_all,12); F['msince']=months_since(M_all); F['nb_lag1']=nb_mean
F['ves_roll12']=roll_mean(M_ves,12)
tot12=np.maximum(roll_mean(M_all,12),1e-6)
for g in ['longline','squid','trawl']: F[f'sh_{g}12']=np.nan_to_num(roll_mean(M_gear[g],12)/tot12)
FEAT_DYN=list(F.keys())

# dist_base_km viene como inf en celdas inalcanzables -> distancia a la base mas cercana SIN filtro de alcance
_db=np.stack([haversine_km(cells.lon.values,cells.lat.values,BASES.loc[b,'lon'],BASES.loc[b,'lat']) for b in BASES.index],1)
cells['dist_base_km']=_db.min(1)
STATIC=['lon','lat','dist_bnd_km','area_km2','dist_base_km','mpa_area_km2','delta']
cells['is_ring']=(cells.zone=='ring').astype(np.float32); cells['no_take_f']=cells.no_take.astype(np.float32)
cells['reg_code']=cells.region.map({'coastal_north':0,'coastal_south':1,'oceanic':2}).astype(np.float32)
STATIC+=['is_ring','no_take_f','reg_code']
FEATURES=STATIC+FEAT_DYN+['month_sin','month_cos','month']

T_START=12                                  # necesitamos 12 meses de historia
ts=np.arange(T_START,NT); n_t=len(ts)
def long(M): return M[:,T_START:].T.ravel()  # orden tiempo-mayor
X=np.empty((n*n_t,len(FEATURES)),np.float32)
for j,f in enumerate(STATIC): X[:,j]=np.tile(cells[f].values.astype(np.float32),n_t)
for j,f in enumerate(FEAT_DYN): X[:,len(STATIC)+j]=np.nan_to_num(long(F[f]))
mo=np.repeat(ts%12+1,n).astype(np.float32); j0=len(STATIC)+len(FEAT_DYN)
X[:,j0]=np.sin(2*np.pi*mo/12); X[:,j0+1]=np.cos(2*np.pi*mo/12); X[:,j0+2]=mo
X=np.nan_to_num(X,nan=0.0,posinf=0.0,neginf=0.0)          # saneo: ningun inf/nan llega al modelo
assert np.isfinite(X).all()
y_all=long(M_all); y_viol=long(M_viol); y_press=long(M_press)
cell_idx=np.tile(np.arange(n),n_t); t_idx=np.repeat(ts,n); year_idx=Y0+t_idx//12
region_idx=np.tile(cells.reg_code.values,n_t)
print(f'panel largo: {X.shape[0]:,} filas x {X.shape[1]} features  ({X.nbytes/1e6:.0f} MB)  en {time.time()-t0:.1f}s')
print('features:',FEATURES)

## B.2 Modelo de riesgo (por región, validación rolling-origin)

In [ ]:
PRETTY={'msince':'months since last active','dist_bnd_km':'signed distance to boundary','nb_lag1':'neighbor effort, t-1','lat':'latitude','lon':'longitude',
        'month_sin':'month (sine)','month_cos':'month (cosine)','month':'calendar month','area_km2':'cell area','roll12':'12-month mean effort','roll3':'3-month mean effort',
        'lag1':'effort, t-1','lag2':'effort, t-2','lag3':'effort, t-3','lag12':'effort, t-12','max12':'12-month max effort','nact12':'active months in last 12',
        'is_ring':'ring cell','no_take_f':'no-take interior','mpa_area_km2':'site area','dist_base_km':'distance to nearest base','sh_squid12':'squid-jigger share',
        'sh_longline12':'longline share','sh_trawl12':'trawl share','ves_roll12':'12-month mean vessels','delta':'cell resolution','reg_code':'region'}
class PoissonGLM:
    """Fallback numpy: Poisson log-link por IRLS con ridge, target escalado y paso amortiguado."""
    def __init__(self,l2=1e-1,it=60): self.l2,self.it=l2,it
    def fit(self,X,y,sample=400000):
        rng=np.random.default_rng(SEED); y=np.asarray(y,np.float64)
        if len(y)>sample:
            pos=np.where(y>0)[0]; neg=np.where(y==0)[0]; keep=np.concatenate([pos,rng.choice(neg,min(len(neg),max(sample-len(pos),0)),replace=False)])
            X,y=X[keep],y[keep]
        self.s_=max(float(y.mean()),1e-6); ys=y/self.s_
        self.mu_=X.mean(0); sd=X.std(0); self.sd_=np.where(sd<1e-6,1.0,sd)
        A=np.hstack([np.ones((len(X),1)),(X-self.mu_)/self.sd_]).astype(np.float64)
        b=np.zeros(A.shape[1]); R=self.l2*np.eye(A.shape[1]); R[0,0]=0
        for _ in range(self.it):
            eta=np.clip(A@b,-15,15); mu=np.exp(eta); z=eta+(ys-mu)/np.maximum(mu,1e-9)
            try: bn=np.linalg.solve(A.T@(A*mu[:,None])+R, A.T@(mu*z))
            except np.linalg.LinAlgError: break
            if not np.all(np.isfinite(bn)): break
            step=bn-b; b=b+0.7*step
            if np.max(np.abs(step))<1e-6: break
        self.b_=b; return self
    def predict(self,X):
        A=np.hstack([np.ones((len(X),1)),(X-self.mu_)/self.sd_]); return self.s_*np.exp(np.clip(A@self.b_,-15,15))
    def feature_importance(self,importance_type=None): return np.abs(self.b_[1:])

def fit_model(Xtr,ytr,Xva=None,yva=None):
    if not HAS_LGB: return PoissonGLM().fit(Xtr,ytr)
    params=dict(objective='tweedie',tweedie_variance_power=1.3,learning_rate=0.04,num_leaves=63,min_child_samples=80,
                feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,lambda_l2=1.0,verbose=-1,seed=SEED,num_threads=os.cpu_count())
    dtr=lgb.Dataset(Xtr,ytr,feature_name=FEATURES)
    if Xva is not None and len(yva)>0:
        dva=lgb.Dataset(Xva,yva,reference=dtr)
        m=lgb.train(params,dtr,num_boost_round=1500,valid_sets=[dva],callbacks=[lgb.early_stopping(60,verbose=False)])
        best=m.best_iteration or 400
        return lgb.train(params,lgb.Dataset(np.vstack([Xtr,Xva]),np.concatenate([ytr,yva]),feature_name=FEATURES),num_boost_round=best)
    return lgb.train(params,dtr,num_boost_round=400)

def poisson_dev(y,mu):
    mu=np.maximum(mu,1e-9)
    with np.errstate(divide='ignore',invalid='ignore'): t=np.where(y>0,y*np.log(y/mu),0.0)
    return float(2*np.mean(t-(y-mu)))
def monthly_metrics(y,mu,tt):
    sp=[];cap=[];auc=[]
    for t in np.unique(tt):
        m=tt==t; yy=y[m]; mm=mu[m]
        if yy.sum()==0: continue
        ra=pd.Series(yy).rank().values; rb=pd.Series(mm).rank().values
        sp.append(np.corrcoef(ra,rb)[0,1] if ra.std()>0 and rb.std()>0 else np.nan)
        k=max(1,int(0.1*len(yy))); cap.append(yy[np.argsort(-mm)[:k]].sum()/yy.sum())
        pos=yy>0; n1,n0=pos.sum(),(~pos).sum()
        auc.append((rb[pos].sum()-n1*(n1+1)/2)/(n1*n0) if n1>0 and n0>0 else np.nan)
    return dict(spearman=float(np.nanmean(sp)),top10_capture=float(np.nanmean(cap)),auc_pos=float(np.nanmean(auc)),poisson_dev=poisson_dev(y,mu))

_rk=['risk_metrics','risk_metrics_by_year','gain_importance','shap_rank']
_c=cache_load('risk')
if _c is None:
    REG={0:'coastal_north',1:'coastal_south',2:'oceanic'}
    r_hat=np.zeros_like(y_all); r_hist=np.nan_to_num(X[:,FEATURES.index('roll12')]); r_lag12=np.nan_to_num(X[:,FEATURES.index('lag12')])
    metrics=[]; models={}; t0=time.time()
    for rc,rname in REG.items():
        rm=region_idx==rc
        for T in TEST_YEARS:
            tr=rm&(year_idx<T-1); va=rm&(year_idx==T-1); te=rm&(year_idx==T)
            if tr.sum()==0 or te.sum()==0: continue
            m=fit_model(X[tr],y_all[tr],X[va],y_all[va]); models[(rname,T)]=m
            mu=np.nan_to_num(np.maximum(m.predict(X[te]),0.0),nan=0.0,posinf=0.0); r_hat[te]=mu
            for name,pred in [('model',mu),('hist12',r_hist[te]),('lag12',r_lag12[te])]:
                metrics.append(dict(region=rname,test_year=T,method=name,n=int(te.sum()),**monthly_metrics(y_all[te],pred,t_idx[te])))
            print(f'  {rname:14s} {T}  n={te.sum():>9,}  spearman={metrics[-3]["spearman"]:.3f}  top10={metrics[-3]["top10_capture"]:.3f}  auc={metrics[-3]["auc_pos"]:.3f}   ({time.time()-t0:.0f}s)')
    met=pd.DataFrame(metrics)
    T2=met.groupby(['region','method']).agg(spearman=('spearman','mean'),top10_capture=('top10_capture','mean'),auc_pos=('auc_pos','mean'),poisson_dev=('poisson_dev','mean')).reset_index()
    print('\n'+T2.round(3).to_string(index=False))
    to_latex(T2,f'{OUT}/T2_risk_metrics.tex','Out-of-sample risk-model performance (rolling origin by year; monthly metrics averaged).','tab:risk',
             fmt={'spearman':'{:.3f}','top10_capture':'{:.3f}','auc_pos':'{:.3f}','poisson_dev':'{:.2f}'})
    met.to_csv(f'{OUT}/risk_metrics_by_year.csv',index=False)
    RESULTS['risk_metrics']=T2.round(3).to_dict('records'); RESULTS['risk_metrics_by_year']=met.round(4).to_dict('records')
    RESULTS['gain_importance']=None

    # ---- importancia / SHAP (ultimo modelo oceanico o el que exista) ----
    mk=[k for k in models if k[0]=='oceanic'] or list(models)
    m=models[mk[-1]]
    try:
        imp=pd.DataFrame({'feature':FEATURES,'gain':m.feature_importance(importance_type='gain')}) if HAS_LGB else pd.DataFrame({'feature':FEATURES,'gain':m.feature_importance()})
    except Exception: imp=pd.DataFrame({'feature':FEATURES,'gain':m.feature_importance()})
    imp=imp.sort_values('gain',ascending=False); imp['gain_%']=100*imp.gain/imp.gain.sum()
    fig,ax=plt.subplots(figsize=(5.4,4.2)); top=imp.head(15)[::-1]
    ax.barh([PRETTY.get(f,f) for f in top.feature],top['gain_%'],color='#2E6FB0'); ax.set_xlabel('Share of total split gain (%)')
    ax.spines[['top','right']].set_visible(False); savefig(fig,'F4_importance')
    if HAS_SHAP and HAS_LGB:
        try:
            te=(region_idx==2)&(year_idx==TEST_YEARS[-1]); idx=np.random.default_rng(SEED).choice(np.where(te)[0],min(20000,te.sum()),replace=False)
            sv=shap.TreeExplainer(m).shap_values(X[idx]); shap.summary_plot(sv,X[idx],feature_names=[PRETTY.get(f,f) for f in FEATURES],show=False,max_display=12)
            np.save(f'{OUT}/shap_mean_abs.npy',np.abs(sv).mean(0)); RESULTS['shap_rank']=[FEATURES[i] for i in np.argsort(-np.abs(sv).mean(0))[:12]]
            plt.gcf().set_size_inches(5.6,3.9); plt.gca().set_xlabel('SHAP value (impact on predicted log effort)',fontsize=9); savefig(plt.gcf(),'F4b_shap'); print('SHAP guardado')
        except Exception as e: print('(shap fallo:',str(e)[:80],')')
    to_latex(imp.head(12)[['feature','gain_%']],f'{OUT}/T5_importance.tex',f'Feature importance of the risk model ({mk[-1][0]} model, test year {mk[-1][1]}; share of total split gain).','tab:imp',fmt={'gain_%':'{:.1f}'})
    print(imp.head(12).round(2).to_string(index=False))
    RESULTS['gain_importance']=imp.head(12)[['feature','gain_%']].round(2).to_dict('records')
    cache_save('risk',dict(r_hat=r_hat,T2=T2,met=met,res={k_:RESULTS.get(k_) for k_ in _rk}))
else:
    r_hat,T2,met=_c['r_hat'],_c['T2'],_c['met']; RESULTS.update(_c['res']); REG={0:'coastal_north',1:'coastal_south',2:'oceanic'}; r_hist=np.nan_to_num(X[:,FEATURES.index('roll12')]); r_lag12=np.nan_to_num(X[:,FEATURES.index('lag12')])


## C. Motor de ruteo (RB-SRP sobre bloques de patrulla)

- Bloque de `BLK_DEG` (0,6°): recompensa y área = suma de sus celdas de mar.
- Tiempo de búsqueda (búsqueda aleatoria): `tau = -A/(W v) ln(1-PMIN)`.
- Salida: parte y vuelve a su base; todo bloque visitado dentro del radio operativo; tiempo total <= `min(T (1-RESERVE), CREW_H)`
  (autonomía neta de reserva de combustible, o límite de servicio de vuelo de la tripulación, lo que sea menor).
- Una salida se construye por inserción + 2-opt + reinserción desde varios arranques, y se mejora con un movimiento de quitar-y-rellenar.
- El mes se planifica con **greedy secuencial**: las salidas se construyen una a una en orden round-robin entre grupos (base, aeronave);
  tras cada salida la recompensa de los bloques visitados se multiplica por `(1-DET)`, `DET = PMIN*PRESENCE`. Luego una **pasada de re-optimización**
  re-planifica cada salida contra todas las demás y acepta solo mejoras.
- Puntaje contra lo realizado: `IA = sum_b H_b [1-(1-DET)^n_b]` — exactamente el objetivo del planificador con `H` en lugar de la predicción.

In [ ]:
# ---------------- blocks ----------------
PL={p:dict(v=float(r.v_kmh),T=float(r.T_h),R=float(r.R_km),W=float(r.W_km)) for p,r in PLATFORMS.iterrows()}
def tb_h(plat): return min(PL[plat]['T']*(1-RESERVE), CREW_H)                                    # time available per sortie (h)
def tau_h(plat): return -BAREA/(PL[plat]['W']*SWEEP_SCALE*PL[plat]['v']*SEARCH_SPEED)*math.log(1-PMIN)   # search time per block (h)
def build_reach():
    """Blocks that a sortie of each (base, platform) may search: within the operating radius and searchable on their own in the time available."""
    global REACH
    REACH={(b,p):np.where((DBASE[b]<=PL[p]['R']*R_SCALE)&(2*DBASE[b]/PL[p]['v']+tau_h(p)<=tb_h(p)))[0] for p in PL for b in COMPAT[p]}
def blk_agg(M):
    out=np.zeros((NB,M.shape[1]),np.float64); np.add.at(out,cells.blk_id.values,M); return out
IS_NT=((cells.zone=='interior')&(cells.no_take==1)).values
def blk_agg_w(M,w=None):
    w=W_NT if w is None else w
    out=np.zeros((NB,M.shape[1])); np.add.at(out,cells.blk_id.values,M*np.where(IS_NT,w,1.0)[:,None]); return out
Rhat=np.zeros((n,NT),np.float32); Rhat[cell_idx,t_idx]=r_hat
HIST=np.nan_to_num(F['roll12'])
JUR=cells.jur.values if cells.jur.notna().all() else None
def build_blocks(deg):
    """Aggregates cells into patrol blocks of `deg` degrees and rebuilds everything that depends on the blocks."""
    global blocks,NB,BLON,BLAT,BAREA,DKM,DBASE,NN_KM,B_all,B_nt,B_press,B_hist,B_rhat,B_oracle,B_jur,blk_mpa
    cells['blk_lon']=(np.floor(cells.lon/deg)*deg+deg/2).round(4); cells['blk_lat']=(np.floor(cells.lat/deg)*deg+deg/2).round(4)
    blocks=(cells.groupby(['blk_lon','blk_lat']).agg(area_km2=('area_km2','sum'),n_cells=('cell_id','size'),lon=('lon','mean'),lat=('lat','mean'),
            region=('region',lambda s:s.mode().iloc[0]),mpa_id=('mpa_id',lambda s:s.mode().iloc[0]),mpa_name=('mpa_name',lambda s:s.mode().iloc[0]),
            mpa_type=('mpa_type',lambda s:s.mode().iloc[0]),reachable=('reachable','max'),n_notake_int=('no_take','sum')).reset_index())
    blocks['blk_id']=np.arange(len(blocks)); cells['blk_id']=cells.drop(columns='blk_id',errors='ignore').merge(blocks[['blk_lon','blk_lat','blk_id']],on=['blk_lon','blk_lat'],how='left').blk_id.values
    NB=len(blocks); BLON,BLAT,BAREA=blocks.lon.values,blocks.lat.values,blocks.area_km2.values
    DKM=haversine_km(BLON[:,None],BLAT[:,None],BLON[None,:],BLAT[None,:]).astype(np.float32)          # block-to-block distances
    DBASE={b:haversine_km(BLON,BLAT,BASES.loc[b,'lon'],BASES.loc[b,'lat']) for b in BASES.index}
    _d=DKM.copy(); np.fill_diagonal(_d,np.inf); NN_KM=_d.min(1).astype(np.float64); del _d             # nearest other block (used by the LP bound)
    build_reach()
    B_all=blk_agg(M_all); B_nt=blk_agg(M_viol); B_press=blk_agg(M_press)
    B_hist=blk_agg_w(HIST); B_rhat=blk_agg_w(Rhat); B_oracle=blk_agg_w(M_all)
    B_jur=blk_agg(M_all*JUR[:,None]) if JUR is not None else None          # realized effort within 200 nm of a coast
    blk_mpa=blocks.mpa_id.values
build_blocks(BLK_DEG)
print(f'blocks: {NB:,} (from {n:,} cells); mean area {blocks.area_km2.mean():,.0f} km2')

# ---------------- single sortie ----------------
def _two_opt(rt,D):
    improved=True; rt=rt[:]
    while improved and len(rt)>4:
        improved=False
        for i in range(1,len(rt)-2):
            for j in range(i+1,len(rt)-1):
                if D[rt[i-1],rt[j]]+D[rt[i],rt[j+1]] < D[rt[i-1],rt[i]]+D[rt[j],rt[j+1]]-1e-9:
                    rt[i:j+1]=rt[i:j+1][::-1]; improved=True
    return rt

def solve_sortie(D,tau,r,Tb,n_seeds=None,swap=None):
    """Selective routing of one sortie. Node 0 is the base; D in hours (symmetric), tau search times, r rewards.
    Insertion by reward per marginal hour, 2-opt and re-insertion, from several starts (the plain construction and constructions seeded
    with each of the n_seeds most valuable blocks); the best route is then improved by a drop-and-refill move.
    Returns (route node list without base, hours, reward)."""
    n_seeds=SORTIE_SEEDS if n_seeds is None else n_seeds; swap=SORTIE_SWAP if swap is None else swap
    m=len(r); free0=np.zeros(m,bool); free0[0]=True; free0[1:]=r[1:]<=0
    def tlen(route): return float(sum(D[route[i],route[i+1]] for i in range(len(route)-1))+tau[route].sum())
    def insert(route,tot,used):
        while True:
            cand=np.where(~used)[0]
            if len(cand)==0: break
            a=np.array(route[:-1]); b=np.array(route[1:])
            dt=D[np.ix_(a,cand)]+tau[cand][None,:]+D[np.ix_(cand,b)].T-D[a,b][:,None]
            feas=(tot+dt)<=Tb+1e-9
            if not feas.any(): break
            score=np.where(feas,r[cand][None,:]/np.maximum(dt,1e-6),-np.inf)
            pi,ci=np.unravel_index(np.argmax(score),score.shape)
            route.insert(pi+1,int(cand[ci])); used[cand[ci]]=True; tot+=dt[pi,ci]
        return route,tot
    def build(seed):
        used=free0.copy(); route=[0,0]; tot=0.0
        if seed is not None:
            route=[0,seed,0]; tot=2*D[0,seed]+tau[seed]; used[seed]=True
        route,tot=insert(route,tot,used)
        if len(route)>4:
            route=_two_opt(route,D); tot=tlen(route); route,tot=insert(route,tot,used)
        return route,tot,float(r[route].sum())
    best=build(None)
    ok=np.where((r>0)&(2*D[0]+tau<=Tb))[0]
    for s in ok[np.argsort(-r[ok],kind='stable')[:n_seeds]]:
        if int(s) in best[0]: continue
        alt=build(int(s))
        if alt[2]>best[2]+1e-12: best=alt
    if swap:                                     # drop one block, refill by insertion, keep the route if its reward increases
        route,tot,cur=best; improved=True
        while improved:
            improved=False
            for pos in range(1,len(route)-1):
                r2=route[:pos]+route[pos+1:]; used=free0.copy(); used[r2]=True; used[route[pos]]=True
                r2,t2=insert(r2,tlen(r2),used)
                if r[r2].sum()>cur+1e-12: route,tot,cur=r2,t2,float(r[r2].sum()); improved=True; break
        best=(route,tot,cur)
    return best[0][1:-1],float(best[1]),best[2]

def plan_sortie(base,plat,rho,pool,tau_all):
    """Sortie of `plat` from `base` over candidate blocks `pool` with rewards rho (indexed by block). Returns (blocks, hours)."""
    if len(pool)==0: return [],0.0
    v=PL[plat]['v']; m=len(pool)+1
    D=np.zeros((m,m)); D[1:,1:]=DKM[np.ix_(pool,pool)]; D[0,1:]=D[1:,0]=DBASE[base][pool]; D/=v
    rt,hrs,_=solve_sortie(D,np.concatenate([[0.0],tau_all[pool]]),np.concatenate([[0.0],rho[pool]]),tb_h(plat))
    return [int(pool[i-1]) for i in rt],hrs

# ---------------- one month: sequential greedy over sortie slots + re-optimization pass ----------------
def month_slots(scen):
    """Groups (base, platform) with their sortie allowance, and the monthly capacity of each base."""
    groups=[]; cap={}
    for base,acs in scen.items():
        na=sum(acs.values())
        if na==0: continue
        cap[base]=int(min(na*SORTIES_PER_AC_MONTH, BASES.loc[base,'sorties_wk']*WEEKS_PER_MONTH))
        for plat,k in acs.items():
            if k>0: groups.append(dict(base=base,plat=plat,left=k*SORTIES_PER_AC_MONTH,ok=REACH[(base,plat)]))
    return groups,cap

def slot_order(scen):
    """Sortie slots in round-robin order over (base, platform) groups, shortest operating radius first within a round."""
    groups,cap=month_slots(scen); gs=sorted(groups,key=lambda g:(PL[g['plat']]['R'],g['base'])); left=[g['left'] for g in gs]; c=dict(cap); sl=[]
    while True:
        added=False
        for i,g in enumerate(gs):
            if left[i]>0 and c[g['base']]>0: sl.append((g['base'],g['plat'])); left[i]-=1; c[g['base']]-=1; added=True
        if not added: break
    return sl

def run_month(scen, rho_month, H_all, H_press, H_nt, kpool=None, keep_routes=False):
    """Plans all sorties of the month for a deployment and scores the plan against realized activity."""
    kpool=K_POOL if kpool is None else int(kpool); DET=PMIN*PRESENCE
    rho0=rho_month.astype(np.float64); nvis=np.zeros(NB); TAU={p:tau_h(p) for p in PL}; plan=[]; visited=set()
    def best(base,plat):
        ok=REACH[(base,plat)]; rho=rho0*(1-DET)**nvis
        r_eff=rho[ok] if COVER_BONUS==0 else rho[ok]*np.where(np.isin(blk_mpa[ok],list(visited)),1.0,1.0+COVER_BONUS)
        top=np.argsort(-r_eff,kind='stable')[:kpool]; pool=ok[top]; rr=np.zeros(NB); rr[pool]=r_eff[top]
        vis,hrs=plan_sortie(base,plat,rr,pool,TAU[plat]); v=np.array(vis,int)
        return v,hrs,float((DET*rho)[v].sum())
    for base,plat in slot_order(scen):                       # construction
        v,hrs,g=best(base,plat)
        nvis[v]+=1; visited.update(blk_mpa[v].tolist()); plan.append([base,plat,v,hrs])
    for _ in range(REOPT_PASSES):                            # re-plan each sortie against all the others; accept only improvements
        for s_ in plan:
            nvis[s_[2]]-=1; old=float((DET*rho0*(1-DET)**nvis)[s_[2]].sum())
            v,hrs,g=best(s_[0],s_[1])
            if g>old+1e-9: s_[2],s_[3]=v,hrs
            nvis[s_[2]]+=1
    plan=[s_ for s_ in plan if len(s_[2])]
    cnt=np.zeros(NB); fh=0.0; curve=[]
    for base,plat,v,hrs in plan:
        cnt[v]+=1; fh+=hrs; curve.append((fh,float((H_all*(1-(1-DET)**cnt)).sum())))
    cov=nvis>0; frac=1-(1-DET)**nvis
    def detected(H): return float((H*frac).sum())
    mp=pd.Series((H_all*frac)[cov]).groupby(blk_mpa[cov]).sum() if cov.any() else pd.Series(dtype=float)
    ia,ia_nt=detected(H_all),detected(H_nt)
    return dict(IA=ia,IA_press=detected(H_press),IA_nt=ia_nt,IA_w=ia+(W_NT-1)*ia_nt,planned=float((rho0*frac).sum()),share_detected=ia/max(H_all.sum(),1e-9),
                flight_h=fh,n_sorties=len(plan),blocks=int(cov.sum()),revisits=float(nvis[cov].mean()) if cov.any() else 0.0,
                gini_mpa=gini(mp.values) if len(mp) else 0.0,nvis=nvis,frac=frac,curve=curve,routes=[(b_,p_,v_.tolist()) for b_,p_,v_,_ in plan] if keep_routes else [])

# ---- engine self-test ----
_t=NT-1; _T=tau_h('P3'); _ok=REACH[('TQM','P3')]; _pool=_ok[np.argsort(-B_oracle[_ok,_t])[:K_POOL]]
_vis,_h=plan_sortie('TQM','P3',B_oracle[:,_t],_pool,_T)
assert _h<=tb_h('P3')+1e-6 and all(DBASE['TQM'][b]<=PL['P3']['R'] for b in _vis)
t0=time.time(); r=run_month(SCENARIOS['S0'],B_rhat[:,_t],B_all[:,_t],B_press[:,_t],B_nt[:,_t])
print(f'self-test sortie P-3 at TQM: {len(_vis)} blocks, {_h:.1f} h | month S0: {r["n_sorties"]} sorties, {r["blocks"]} blocks, {r["flight_h"]:.0f} flight h, IA={r["IA"]:,.0f} h ({100*r["share_detected"]:.1f}%), {time.time()-t0:.2f}s')
_s0=time.time()-t0; t0=time.time(); r6=run_month(SCENARIOS['S6'],B_rhat[:,_t],B_all[:,_t],B_press[:,_t],B_nt[:,_t]); print(f'month S6: {r6["n_sorties"]} sorties, IA={r6["IA"]:,.0f} h, {time.time()-t0:.2f}s')
RESULTS['plan_seconds']=dict(S0=round(_s0,3),S6=round(time.time()-t0,3))

## D. Backtest fuera de muestra

In [ ]:
test_t=[t for t in range(T_START,NT) if Y0+t//12 in TEST_YEARS]
def reward_vec(mt,t,rng=None):
    if mt=='risk': return B_rhat[:,t]
    if mt=='hist': return B_hist[:,t]
    if mt=='uniform': return np.ones(NB)
    if mt=='oracle': return B_oracle[:,t]
    return rng.random(NB)
def evaluate(scen,mt,t,rng=None,keep_routes=False,reward=None):
    res=run_month(scen,reward_vec(mt,t,rng) if reward is None else reward,B_all[:,t],B_press[:,t],B_nt[:,t],kpool=(NB if mt=='uniform' else None),keep_routes=keep_routes)
    res['IA_jur']=float((B_jur[:,t]*res['frac']).sum()) if B_jur is not None else np.nan
    return res
_c=cache_load('backtest')
if _c is None:
    rows=[]; curves={}; t0=time.time(); rng=np.random.default_rng(SEED)
    jobs=[(sc,mt,t) for sc in SCENARIOS for mt in (METHODS_FULL if sc in FULL_METHOD_SCEN else METHODS_LIGHT) for t in test_t]
    for k,(sc,mt,t) in enumerate(jobs,1):
        for rep in range(N_RANDOM if mt=='random' else 1):
            res=evaluate(SCENARIOS[sc],mt,t,rng)
            rows.append(dict(scenario=sc,method=mt,t=t,year=Y0+t//12,month=t%12+1,rep=rep,**{k_:v for k_,v in res.items() if k_ not in ('curve','routes','nvis','frac')}))
            if sc in ('S0','S5') and rep==0: curves.setdefault((sc,mt),[]).append(res['curve'])
        if k%100==0 or k==len(jobs): print(f'  [{k}/{len(jobs)}] {sc} {mt:8s}  ETA {(time.time()-t0)/k*(len(jobs)-k)/60:4.1f} min')
    cache_save('backtest',(rows,curves))
else: rows,curves=_c
bt=pd.DataFrame(rows); bt['IA_per_h']=bt.IA/bt.flight_h.replace(0,np.nan)
bt.to_csv(f'{OUT}/backtest_monthly.csv',index=False)
print(f'backtest: {len(bt):,} rows in {(time.time()-t0)/60:.1f} min')

In [ ]:
BM=bt.groupby(['scenario','method','t'],as_index=False).mean(numeric_only=True)          # average random replications
S=BM.groupby(['scenario','method']).agg(IA=('IA','mean'),IA_press=('IA_press','mean'),IA_nt=('IA_nt','mean'),IA_w=('IA_w','mean'),IA_jur=('IA_jur','mean'),share=('share_detected','mean'),
        flight_h=('flight_h','mean'),sorties=('n_sorties','mean'),blocks=('blocks','mean'),revisits=('revisits','mean'),gini=('gini_mpa','mean')).reset_index()
S['IA_per_h']=S.IA/S.flight_h
def ser(sc,mt,col='IA'): return BM[(BM.scenario==sc)&(BM.method==mt)].set_index('t')[col].reindex(test_t)
def get(sc,mt): return S[(S.scenario==sc)&(S.method==mt)].iloc[0]
NT_REALIZED_PM=float(B_nt[:,test_t].sum(0).mean()); RESULTS['nt_realized_per_month_test']=NT_REALIZED_PM
RESULTS['all_realized_per_month_test']=float(B_all[:,test_t].sum(0).mean()); RESULTS['jur_realized_per_month_test']=float(B_jur[:,test_t].sum(0).mean()) if B_jur is not None else None

# ---- T3: methods under S0 and S5 ----
T3=[]
for sc in FULL_METHOD_SCEN:
    for mt in METHODS_FULL:
        r=get(sc,mt); o=get(sc,'oracle')
        d=dict(scenario=sc,method=mt,IA=r.IA,IA_jur=r.IA_jur,share=r.share,IA_nt=r.IA_nt,IA_w=r.IA_w,IA_per_h=r.IA_per_h,of_oracle=r.IA_w/o.IA_w,flight_h=r.flight_h,blocks=r.blocks,revisits=r.revisits,gini=r.gini,
               wins=np.nan,gain_pct=np.nan,ci_lo=np.nan,ci_hi=np.nan,p=np.nan)
        if mt not in ('risk','oracle'):
            a,b=ser(sc,'risk').values,ser(sc,mt).values
            d['wins']=int((a>b).sum()); d['gain_pct']=100*(a.mean()/b.mean()-1); d['ci_lo'],d['ci_hi']=boot_ratio_ci(a,b); d['p']=wilcoxon_paired(a,b,'greater')[1]
        T3.append(d)
T3=pd.DataFrame(T3)
for sc in FULL_METHOD_SCEN: T3.loc[T3.scenario==sc,'p_holm']=holm(T3.loc[T3.scenario==sc,'p'].values)
print(T3.round(4).to_string(index=False))
Mf=BM[(BM.scenario=='S0')&(BM.method.isin(['risk','hist','uniform','random']))].pivot(index='t',columns='method',values='IA')
fr=friedman_nemenyi(Mf.values,list(Mf.columns)); print('Friedman S0:',fr); RESULTS['friedman_S0']=fr

# ---- T4: scenarios under risk-based planning, paired against S0 ----
T4=[]
for sc in SCENARIOS:
    r=get(sc,'risk'); o=get(sc,'oracle'); h=get(sc,'hist'); a=ser(sc,'risk').values; b=ser('S0','risk').values
    av=ser(sc,'risk','IA_nt').values; bv=ser('S0','risk','IA_nt').values
    d=dict(scenario=sc,aircraft=sum(sum(x.values()) for x in SCENARIOS[sc].values()),IA=r.IA,IA_jur=r.IA_jur,delta_pct=100*(r.IA/get('S0','risk').IA-1),ci_lo=np.nan,ci_hi=np.nan,p=np.nan,
           share=r.share,of_oracle=r.IA_w/o.IA_w,IA_nt=r.IA_nt,nt_capture=r.IA_nt/NT_REALIZED_PM,p_nt=np.nan,flight_h=r.flight_h,sorties=r.sorties,IA_per_h=r.IA_per_h,gini=r.gini,
           IA_hist=h.IA,IA_oracle=o.IA,risk_vs_hist_pct=100*(r.IA/h.IA-1))
    if sc!='S0':
        d['ci_lo'],d['ci_hi']=boot_ratio_ci(a,b); d['p']=wilcoxon_paired(a,b,'greater')[1]; d['p_nt']=wilcoxon_paired(av,bv,'greater')[1]
    T4.append(d)
T4=pd.DataFrame(T4); T4['p_holm']=holm(T4.p.values); T4['p_nt_holm']=holm(T4.p_nt.values)
print('\n'+T4.round(4).to_string(index=False))
by_year=BM[BM.method.isin(['risk','oracle'])].groupby(['scenario','method','year']).agg(IA=('IA','mean'),share=('share_detected','mean'),IA_nt=('IA_nt','mean')).reset_index()
by_year.to_csv(f'{OUT}/backtest_by_year.csv',index=False)
T3.to_csv(f'{OUT}/T3_methods.csv',index=False); T4.to_csv(f'{OUT}/T4_scenarios.csv',index=False)
RESULTS['T3']=T3.round(5).to_dict('records'); RESULTS['T4']=T4.round(5).to_dict('records')
RESULTS['oracle_S6_vs_S5_pct']=float(100*(get('S6','oracle').IA/get('S5','oracle').IA-1))

# ---- F5: cumulative intercepted activity vs flight hours (S0 and S5) ----
fig,axs=plt.subplots(1,2,figsize=(7.0,2.75),sharey=True); crow=[]
for ax,sc in zip(axs,['S0','S5']):
    for mt,col,ls in [('oracle',C_ORA,':'),('risk',C_RISK,'-'),('hist',C_HIST,'--'),('uniform',C_UNI,'-.')]:
        cs=[c for c in curves.get((sc,mt),[]) if c]
        if not cs: continue
        grid=np.linspace(0,max(c[-1][0] for c in cs),120)
        ya=np.mean([np.interp(grid,[0]+[p[0] for p in c],[0]+[p[1] for p in c]) for c in cs],0)
        ax.plot(grid,ya/1000,color=col,ls=ls,lw=1.6,label=M_LABEL[mt]); crow+=[dict(scenario=sc,method=mt,flight_h=g_,IA=y_) for g_,y_ in zip(grid,ya)]
    ax.set_title(SC_LABEL[sc],fontsize=9); ax.set_xlabel('Cumulative flight hours in the month'); ax.spines[['top','right']].set_visible(False); ax.grid(alpha=.25)
axs[0].set_ylabel('Detected activity\n(thousand hours)')
hs,ls_=axs[0].get_legend_handles_labels()
fig.legend(hs,ls_,loc='lower center',ncol=4,frameon=False,fontsize=8,handlelength=2.6,columnspacing=2.2,bbox_to_anchor=(0.53,-0.02))   # one legend for both panels, below the axes
plt.tight_layout(rect=(0,0.07,1,1)); savefig(fig,'F5_detection_effort'); pd.DataFrame(crow).to_csv(f'{OUT}/F5_curves.csv',index=False)

# ---- F7: example month, sorties planned under S0 and S5 (risk-based) over the effort realized in that month ----
t_ex=test_t[-10]                                           # March of the last test year
fig,axs=plt.subplots(1,2,figsize=(7.1,3.75)); H=B_all[:,t_ex]; pos=H>0
_,_,xe,ye,GB=raster(blocks.blk_lon.values[pos],blocks.blk_lat.values[pos],np.full(pos.sum(),BLK_DEG),val=H[pos],step=BLK_DEG,shape=(90,117))
for k_,(ax,sc) in enumerate(zip(axs,['S0','S5'])):
    res=evaluate(SCENARIOS[sc],'risk',t_ex,keep_routes=True); draw_base(ax,label_left=(k_==0))
    pm=ax.pcolormesh(xe,ye,np.ma.masked_invalid(GB),cmap='YlOrRd',norm=LogNorm(vmin=1,vmax=3000),zorder=1.6,rasterized=True)
    draw_mpas(ax,fill=False,lw=0.5,mu_ls=(0,(3,1.5)))
    for base,plat,vis in res['routes']:
        xs=[BASES.loc[base,'lon']]+list(BLON[vis])+[BASES.loc[base,'lon']]; ys=[BASES.loc[base,'lat']]+list(BLAT[vis])+[BASES.loc[base,'lat']]
        ax.plot(xs,merc(ys),lw=0.55,color=C_P3 if plat=='P3' else C_C295,alpha=0.75,zorder=4,solid_joinstyle='round')
    draw_bases(ax,which=list(SCENARIOS[sc]),names=False)
    panel_title(ax,f'({"ab"[k_]}) {SC_LABEL[sc]}: {res["n_sorties"]} sorties, {100*res["share_detected"]:.0f}% detected')
axs[0].legend(handles=[Line2D([],[],color=C_P3,lw=1,label='P-3ACH sortie'),Line2D([],[],color=C_C295,lw=1,label='C-295 sortie'),Line2D([],[],marker='s',ms=3.6,color='k',ls='',label='Base in use'),
      Line2D([],[],color=C_MPAE,lw=0.6,label='No-take site'),Line2D([],[],color=C_MPAE,lw=0.6,ls=(0,(3,1.5)),label='Multiple-use site')],
      loc='lower left',fontsize=5.6,frameon=True,framealpha=0.95,edgecolor='#888888',borderpad=0.5,labelspacing=0.3,handlelength=1.8).set_zorder(15)
cax=axs[1].inset_axes([0.05,0.075,0.44,0.028]); cb=fig.colorbar(pm,cax=cax,orientation='horizontal'); cb.outline.set_linewidth(0.4)
cb.set_ticks([1,10,100,1000]); cb.set_ticklabels(['1','10','100','1,000']); cax.tick_params(labelsize=5.6,length=1.5,width=0.4)
cax.set_title('Effort realized in the month (hours per block)',fontsize=5.8,pad=2)
plt.subplots_adjust(wspace=0.04,left=0.06,right=0.995,top=0.99,bottom=0.07)
fig.savefig(f'{OUT}/F7_example_month.pdf',bbox_inches='tight',dpi=400); fig.savefig(f'{OUT}/F7_example_month.png',dpi=200,bbox_inches='tight'); plt.close(fig)
RESULTS['example_month']=f'{Y0+t_ex//12}-{t_ex%12+1:02d}'
print('F5, F7 saved')

## E. Cota superior LP del óptimo mensual

Relajación lineal válida del RB-SRP con recompensa de información perfecta. Cada salida se clasifica por el nivel radial de su
bloque más cercano (`a`) y más lejano (`c`); el tiempo de tránsito se acota por abajo con `2 r_{c-1}` y con
`2 r_{a-1}` + saltos al vecino más cercano. El valor de la LP es una **cota superior certificada** del óptimo, así que
`heurística / cota` es una cota inferior de la calidad de la solución.

In [ ]:
def lp_upper_bound(scen, rho, L=None, nmax=None, far_c=True, near_c=True):
    """Upper bound on the optimal monthly value of the RB-SRP with block rewards rho (same units as IA_w)."""
    L=L or LP_LEVELS; DET=PMIN*PRESENCE; groups,cap=month_slots(scen)
    if nmax is None: nmax=6 if DET>=0.6 else int(max(6,math.ceil(math.log(0.01)/math.log(1-DET))))   # searches per block before the last, unbounded, level: (1-DET)^nmax <= 1%
    pos=np.where(rho>0)[0]; pos=pos[np.isin(pos,np.unique(np.concatenate([g['ok'] for g in groups])))]
    if len(pos)==0: return 0.0
    bidx={b:i for i,b in enumerate(pos)}; nbk=len(pos)
    cols=[]; A_r=[]; A_c=[]; A_v=[]; rhs=[]; nrow=0
    def add_row(cs,vs,b):
        nonlocal nrow
        A_r.extend([nrow]*len(cs)); A_c.extend(cs); A_v.extend(vs); rhs.append(b); nrow+=1
    nvar=0; ycols=[[] for _ in range(nbk)]; base_rows={b:[] for b in cap}
    for g in groups:
        p=PL[g['plat']]; Tb=tb_h(g['plat']); v=p['v']; tau=tau_h(g['plat'])
        bl=pos[np.isin(pos,g['ok'])]
        if len(bl)==0: continue
        d=DBASE[g['base']][bl]; nnmax=NN_KM[bl].max()
        thr=np.unique(np.quantile(d,np.linspace(0,1,L+1)))                                   # r_0 = nearest block < r_1 < ... ; level i covers (r_{i-1}, r_i]
        if len(thr)<2: thr=np.array([thr[0],thr[0]+1.0])
        lev=np.clip(np.searchsorted(thr,d,side='left'),1,len(thr)-1)
        gvars=[]
        for a in range(1,len(thr)):
            for c in range(a,len(thr)):
                far=Tb-2*thr[c-1]/v; near=Tb-2*thr[a-1]/v+nnmax/v
                sel=np.where((lev>=a)&(lev<=c))[0]
                if far<=0 or near<=0 or len(sel)==0: continue
                nv=nvar; nvar+=1; gvars.append(nv); ys=np.arange(nvar,nvar+len(sel)); nvar+=len(sel)
                for j,yv in zip(sel,ys):
                    ycols[bidx[bl[j]]].append(int(yv)); add_row([int(yv),nv],[1.0,-1.0],0.0)          # y <= n  (a sortie searches a block at most once)
                if far_c: add_row(list(ys)+[nv],list(tau[bl[sel]])+[-far],0.0)                        # search time <= on-station time (far leg)
                if near_c: add_row(list(ys)+[nv],list(tau[bl[sel]]+NN_KM[bl[sel]]/v)+[-near],0.0)     # search + hops <= time net of near leg
        if gvars: add_row(gvars,[1.0]*len(gvars),float(g['left'])); base_rows[g['base']].extend(gvars)
    for b,gv in base_rows.items():
        if gv: add_row(gv,[1.0]*len(gv),float(cap[b]))
    w0=nvar; nvar+=nbk*nmax
    cobj=np.zeros(nvar); ub=np.full(nvar,np.inf)
    E_r=[];E_c=[];E_v=[]
    for i in range(nbk):
        for k in range(nmax):
            cobj[w0+i*nmax+k]=-rho[pos[i]]*DET*(1-DET)**k
            if k<nmax-1: ub[w0+i*nmax+k]=1.0
            E_r.append(i); E_c.append(w0+i*nmax+k); E_v.append(1.0)
        for yv in ycols[i]: E_r.append(i); E_c.append(yv); E_v.append(-1.0)
    A=sp.csr_matrix((A_v,(A_r,A_c)),shape=(nrow,nvar)); E=sp.csr_matrix((E_v,(E_r,E_c)),shape=(nbk,nvar))
    res=linprog(cobj,A_ub=A,b_ub=np.array(rhs),A_eq=E,b_eq=np.zeros(nbk),bounds=list(zip(np.zeros(nvar),ub)),method='highs')
    assert res.status==0, res.message
    return float(-res.fun)

LP_SCEN=['S0','S5']; LP_T=test_t[::4] if FAST_MODE else test_t
lrows=[]; t0=time.time()
for sc in LP_SCEN:
    for t in LP_T:
        ubd=lp_upper_bound(SCENARIOS[sc],B_oracle[:,t])
        lrows.append(dict(scenario=sc,t=t,UB=ubd,oracle=float(ser(sc,'oracle','IA_w')[t]),risk=float(ser(sc,'risk','IA_w')[t]),hist=float(ser(sc,'hist','IA_w')[t])))
    print(f'  LP bound {sc}: {len(LP_T)} months, {(time.time()-t0)/60:.1f} min')
lpb=pd.DataFrame(lrows); lpb.to_csv(f'{OUT}/lp_bound_monthly.csv',index=False)
assert (lpb.oracle<=lpb.UB*(1+1e-6)).all(), 'heuristic above the upper bound: check the relaxation'
T7=lpb.groupby('scenario').apply(lambda d: pd.Series(dict(months=len(d),UB=d.UB.mean(),oracle_of_UB=d.oracle.sum()/d.UB.sum(),oracle_of_UB_min=(d.oracle/d.UB).min(),
        oracle_of_UB_median=(d.oracle/d.UB).median(),risk_of_UB=d.risk.sum()/d.UB.sum(),hist_of_UB=d['hist'].sum()/d.UB.sum()))).reset_index()
print(T7.round(4).to_string(index=False)); T7.to_csv(f'{OUT}/T7_lp_bound.csv',index=False); RESULTS['T7']=T7.round(5).to_dict('records')

## F. Benchmark exacto de la heurística de una salida

Instancias reales: mes de test, base y aeronave al azar; los `m` bloques de mayor recompensa predicha dentro del radio.
Óptimo por programación dinámica sobre subconjuntos (Held–Karp con presupuesto de tiempo).

In [ ]:
def exact_sortie(D,tau,r,Tb):
    """Optimal reward of one sortie by dynamic programming over subsets (node 0 = base)."""
    m=len(r)-1; full=1<<m; Dn=D[1:,1:]; d0=D[0,1:]; tn=tau[1:]; rn=r[1:]
    dp=np.full((full,m),np.inf); rew=np.zeros(full)
    for j in range(m): dp[1<<j,j]=d0[j]+tn[j]
    for Sx in range(1,full):
        low=(Sx&-Sx).bit_length()-1; rew[Sx]=rew[Sx&(Sx-1)]+rn[low]
    best=0.0; inS=((np.arange(full)[:,None]>>np.arange(m)[None,:])&1).astype(bool)
    for Sx in range(1,full):
        row=dp[Sx]
        if not np.isfinite(row).any(): continue
        if (row+d0).min()<=Tb+1e-9 and rew[Sx]>best: best=rew[Sx]
        ext=(row[:,None]+Dn+tn[None,:]).min(0)                     # cheapest arrival at each k from the best last node
        for k in np.where(~inS[Sx]&(ext+d0<=Tb+1e-9))[0]:
            S2=Sx|(1<<k)
            if ext[k]<dp[S2,k]: dp[S2,k]=ext[k]
    return float(best)

rng=np.random.default_rng(SEED+1); hq=[]; t0=time.time()
SIZES=[8,11] if FAST_MODE else [8,11,14]; N_INST=20 if FAST_MODE else 100
for plat in PL:
    for m_ in SIZES:
        k=0; tries=0
        while k<N_INST and tries<20000:
            tries+=1
            t=int(rng.choice(test_t)); base=str(rng.choice(COMPAT[plat])); ok=REACH[(base,plat)]; rho=B_rhat[:,t]
            top=ok[np.argsort(-rho[ok],kind='stable')[:100]]
            if len(top)<m_: continue
            pool=rng.choice(top,m_,replace=False)
            if rho[pool].min()<=0: continue
            v=PL[plat]['v']; D=np.zeros((m_+1,m_+1)); D[1:,1:]=DKM[np.ix_(pool,pool)]; D[0,1:]=D[1:,0]=DBASE[base][pool]; D/=v
            tau=np.concatenate([[0.0],tau_h(plat)[pool]]); rr=np.concatenate([[0.0],rho[pool]]); Tb=tb_h(plat)
            opt=exact_sortie(D,tau,rr,Tb)
            if opt<=0 or opt>=rr.sum()-1e-9: continue                 # keep only instances in which endurance binds
            h1=solve_sortie(D,tau,rr,Tb,n_seeds=1,swap=False)[2]; h2=solve_sortie(D,tau,rr,Tb)[2]          # screening and full configurations
            assert h2<=opt*(1+1e-9)
            hq.append(dict(plat=plat,m=m_,base=base,t=t,opt=opt,ratio_single=h1/opt,ratio=h2/opt)); k+=1
    print(f'  {plat}: done ({(time.time()-t0)/60:.1f} min)')
hq=pd.DataFrame(hq); hq.to_csv(f'{OUT}/sortie_benchmark.csv',index=False)
T8=hq.groupby(['plat','m']).agg(instances=('ratio','size'),mean_ratio=('ratio','mean'),min_ratio=('ratio','min'),optimal_pct=('ratio',lambda s:100*(s>1-1e-9).mean()),mean_ratio_single=('ratio_single','mean')).reset_index()
T8=pd.concat([T8,pd.DataFrame([dict(plat='all',m=0,instances=len(hq),mean_ratio=hq.ratio.mean(),min_ratio=hq.ratio.min(),optimal_pct=100*(hq.ratio>1-1e-9).mean(),mean_ratio_single=hq.ratio_single.mean())])])
print(T8.round(4).to_string(index=False)); T8.to_csv(f'{OUT}/T8_sortie_benchmark.csv',index=False); RESULTS['T8']=T8.round(5).to_dict('records')

## G. Enumeración de basamientos de la flota actual

Todas las asignaciones de 2 P-3ACH y 3 C-295 a bases compatibles (6 × 56 = 336). Cada una se evalúa en el **año de selección**
(anterior a los años de test) planificando con el mapa histórico, que es la información disponible en ese momento.
Luego **todas** se validan fuera de muestra en los meses de test con planificación por riesgo. La enumeración usa una
configuración rápida de la heurística (lista de 150 candidatos, un arranque sembrado, sin quitar-y-rellenar).

In [ ]:
def deployments():
    out=[]
    for cp in itertools.combinations_with_replacement(COMPAT['P3'],FLEET['P3']):
        for cc in itertools.combinations_with_replacement(COMPAT['C295'],FLEET['C295']):
            d={}
            for b in cp: d.setdefault(b,{}).setdefault('P3',0); d[b]['P3']+=1
            for b in cc: d.setdefault(b,{}).setdefault('C295',0); d[b]['C295']+=1
            out.append(d)
    return out
def dep_name(d): return ' + '.join(f'{b}:'+','.join(f'{k}{p.replace("P3","P").replace("C295","C")}' for p,k in sorted(v.items(),reverse=True)) for b,v in sorted(d.items()))
def same(d1,d2): return json.dumps(d1,sort_keys=True)==json.dumps(d2,sort_keys=True)
DEPS=deployments(); print(len(DEPS),'deployments')
sel_t=[t for t in range(T_START,NT) if Y0+t//12==SEL_YEAR]
if FAST_MODE: sel_t=sel_t[::3]
_K,_SS,_SW=K_POOL,SORTIE_SEEDS,SORTIE_SWAP; K_POOL,SORTIE_SEEDS,SORTIE_SWAP=150,1,False      # fast screening configuration for the enumeration (restored below)
erows=cache_load('enum_sel')
if erows is None:
    erows=[]; t0=time.time()
    for i,d in enumerate(DEPS):
        acc=dict(IA=0.0,IA_nt=0.0,IA_w=0.0,flight_h=0.0)
        for t in sel_t:
            res=evaluate(d,'hist',t)
            for k_ in acc: acc[k_]+=res[k_]/len(sel_t)
        erows.append(dict(dep=i,name=dep_name(d),**acc))
        if (i+1)%48==0: print(f'  [{i+1}/{len(DEPS)}]  ETA {(time.time()-t0)/(i+1)*(len(DEPS)-i-1)/60:.1f} min')
    cache_save('enum_sel',erows)
en=pd.DataFrame(erows); nt_sel=float(B_nt[:,sel_t].sum(0).mean()); en['nt_capture']=en.IA_nt/max(nt_sel,1e-9)
en['rank_IA']=en.IA.rank(ascending=False).astype(int); en['rank_w']=en.IA_w.rank(ascending=False).astype(int)
o=en.sort_values(['IA','IA_nt'],ascending=False); best=-1; par=[]
for _,r in o.iterrows():
    if r.IA_nt>best+1e-9: par.append(int(r.dep)); best=r.IA_nt
en['pareto']=en.dep.isin(par)
for s_ in ['S0','S1','S2','S3','S4','S5']: en.loc[[i for i,d in enumerate(DEPS) if same(d,SCENARIOS[s_])],'scenario']=s_
i_star=int(en.sort_values('IA_w',ascending=False).iloc[0].dep); en.loc[i_star,'star']=True
en.to_csv(f'{OUT}/basing_enumeration.csv',index=False)
print(en.sort_values('IA_w',ascending=False).head(8)[['name','IA','IA_nt','nt_capture','rank_IA','scenario']].to_string(index=False))
print('named scenarios:\n',en[en.scenario.notna()][['scenario','name','IA','nt_capture','rank_IA','rank_w']].to_string(index=False))

# ---- out-of-sample validation of every deployment (risk-based planning, test months) ----
t0=time.time(); OOS_IA=np.zeros((len(DEPS),len(test_t))); OOS_NT=np.zeros((len(DEPS),len(test_t)))
oos_ids=range(len(DEPS)) if not FAST_MODE else list(dict.fromkeys([i_star]+par+[int(i) for i in en[en.scenario.notna()].dep]+list(np.random.default_rng(SEED+2).choice(len(DEPS),12,replace=False))))
_c=cache_load('enum_oos')
if _c is None:
    for j,i in enumerate(oos_ids):
        for q,t in enumerate(test_t):
            res=evaluate(DEPS[i],'risk',t); OOS_IA[i,q]=res['IA']; OOS_NT[i,q]=res['IA_nt']
        if (j+1)%48==0: print(f'  out-of-sample [{j+1}/{len(oos_ids)}]  ETA {(time.time()-t0)/(j+1)*(len(oos_ids)-j-1)/60:.1f} min')
    cache_save('enum_oos',(OOS_IA,OOS_NT))
else: OOS_IA,OOS_NT=_c
K_POOL,SORTIE_SEEDS,SORTIE_SWAP=_K,_SS,_SW
ids=list(oos_ids); en['IA_test']=np.nan; en['nt_capture_test']=np.nan
en.loc[ids,'IA_test']=OOS_IA[ids].mean(1); en.loc[ids,'nt_capture_test']=OOS_NT[ids].mean(1)/NT_REALIZED_PM
en['rank_test']=en.IA_test.rank(ascending=False); en.to_csv(f'{OUT}/basing_enumeration.csv',index=False)
np.save(f'{OUT}/basing_oos_IA.npy',OOS_IA)
oos=en.loc[ids]; rho_s=stats.spearmanr(oos.IA,oos.IA_test).correlation
i0=int(en[en.scenario=='S0'].dep.iloc[0]); i5=int(en[en.scenario=='S5'].dep.iloc[0]); i_post=int(oos.sort_values('IA_test',ascending=False).iloc[0].dep)
star_ia,star_nt,b0,b5=OOS_IA[i_star],OOS_NT[i_star],OOS_IA[i0],OOS_IA[i5]
top=en.sort_values('IA_w',ascending=False).head(10)[['name','IA','nt_capture','IA_test','nt_capture_test','rank_test','scenario']]
print(top.round(3).to_string(index=False)); top.to_csv(f'{OUT}/T9_basing_top10.csv',index=False)
STAR=dict(name=en.loc[i_star,'name'],is_S5=bool(i_star==i5),IA_sel=float(en.loc[i_star,'IA']),nt_capture_sel=float(en.loc[i_star,'nt_capture']),IA_test=float(star_ia.mean()),nt_capture_test=float(star_nt.mean()/NT_REALIZED_PM),
          rank_test=int(en.loc[i_star,'rank_test']),IA_test_S0=float(b0.mean()),IA_test_S5=float(b5.mean()),nt_capture_test_S5=float(OOS_NT[i5].mean()/NT_REALIZED_PM),
          delta_vs_S0_pct=float(100*(star_ia.mean()/b0.mean()-1)),ci_vs_S0=boot_ratio_ci(star_ia,b0),p_vs_S0=wilcoxon_paired(star_ia,b0,'greater')[1],
          delta_vs_S5_pct=float(100*(star_ia.mean()/b5.mean()-1)),ci_vs_S5=boot_ratio_ci(star_ia,b5),
          expost_best=en.loc[i_post,'name'],IA_test_expost_best=float(en.loc[i_post,'IA_test']),regret_pct=float(100*(1-star_ia.mean()/en.loc[i_post,'IA_test'])),
          rank_S0=int(en.loc[i0,'rank_IA']),rank_S5=int(en.loc[i5,'rank_IA']),rank_S0_w=int(en.loc[i0,'rank_w']),rank_S5_w=int(en.loc[i5,'rank_w']),
          rank_test_S0=int(en.loc[i0,'rank_test']),rank_test_S5=int(en.loc[i5,'rank_test']),
          n_deployments=len(DEPS),n_pareto=len(par),pareto=[en.loc[i,'name'] for i in par],spearman_sel_test=float(rho_s),n_oos=len(ids),
          IA_sel_min=float(en.IA.min()),IA_sel_median=float(en.IA.median()),IA_sel_max=float(en.IA.max()),IA_test_min=float(oos.IA_test.min()),IA_test_median=float(oos.IA_test.median()),IA_test_max=float(oos.IA_test.max()),
          share_below_S0=float((en.IA<en.loc[i0,'IA']).mean()),share_below_S0_test=float((oos.IA_test<en.loc[i0,'IA_test']).mean()))
RESULTS['basing']=STAR; print(json.dumps(STAR,indent=1,default=str))

# ---- F6: all deployments in the plane of the two objectives (selection year and test years) ----
fig,axs=plt.subplots(1,2,figsize=(7.0,3.1),sharey=True)
for ax,(xc,yc,ttl) in zip(axs,[('IA','nt_capture',f'Selection year ({SEL_YEAR}), historical map'),('IA_test','nt_capture_test',f'Test years ({TEST_YEARS[0]}–{TEST_YEARS[-1]}), risk model')]):
    e=en[en[xc].notna()]
    ax.scatter(e[xc]/1000,100*e[yc],s=8,c='#B8C0C8',linewidths=0,label='all deployments',zorder=2)
    if xc=='IA':
        pp=e[e.pareto].sort_values(xc); ax.plot(pp[xc]/1000,100*pp[yc],color='k',lw=0.9,ls='--',marker='.',ms=4,label='Pareto frontier',zorder=3)
    for s_,mk in zip(['S0','S1','S2','S3','S4','S5'],['X','v','^','D','s','o']):
        r=e[e.scenario==s_].iloc[0]; ax.scatter(r[xc]/1000,100*r[yc],s=40,marker=mk,c=C_NT if s_=='S0' else C_MU,edgecolors='k',linewidths=0.6,zorder=5,label=SC_LABEL[s_])
    r=e.loc[i_star]; ax.scatter(r[xc]/1000,100*r[yc],s=130,marker='*',c='#F1C40F',edgecolors='k',linewidths=0.6,zorder=6,label='S* (selected)')
    ax.set_title(ttl,fontsize=8.5); ax.set_xlabel('Detected activity (10$^3$ h per month)'); ax.spines[['top','right']].set_visible(False); ax.grid(alpha=.25)
axs[0].set_ylabel('No-take effort detected (%)')
axs[1].legend(frameon=False,fontsize=7,loc='center left',bbox_to_anchor=(1.0,0.5))
plt.tight_layout(); savefig(fig,'F6_basing'); print('F6 saved')

## H. Sensibilidad a los supuestos (S0 y S5, planificación por riesgo)

In [ ]:
SENS=[('baseline',{}),('PMIN=0.6',{'PMIN':0.6}),('PMIN=0.9',{'PMIN':0.9}),('presence=0.5',{'PRESENCE':0.5}),('reserve=0.25',{'RESERVE':0.25}),
      ('sorties=6',{'SORTIES_PER_AC_MONTH':6}),('sorties=14',{'SORTIES_PER_AC_MONTH':14}),('IPC group tier',{'IPC_WK':5}),
      ('weight=1',{'W_NT':1.0}),('weight=20',{'W_NT':20.0}),('cover bonus=0.5',{'COVER_BONUS':0.5}),
      ('radius -25%',{'R_SCALE':0.75}),('radius +25%',{'R_SCALE':1.25}),('no radius limit',{'R_SCALE':99.0}),('sweep -25%',{'SWEEP_SCALE':0.75}),('sweep +25%',{'SWEEP_SCALE':1.25}),('search speed 80%',{'SEARCH_SPEED':0.8})]
if FAST_MODE: SENS=SENS[:2]+SENS[11:14]
_g=dict(PMIN=PMIN,PRESENCE=PRESENCE,RESERVE=RESERVE,SORTIES_PER_AC_MONTH=SORTIES_PER_AC_MONTH,W_NT=W_NT,COVER_BONUS=COVER_BONUS,IPC_WK=float(BASES.loc['IPC','sorties_wk']),R_SCALE=R_SCALE,SWEEP_SCALE=SWEEP_SCALE,SEARCH_SPEED=SEARCH_SPEED)
def _apply(ch):
    for k_,v_ in {**_g,**ch}.items():
        if k_=='IPC_WK': BASES.loc['IPC','sorties_wk']=v_
        else: globals()[k_]=v_
    build_reach()
srows=cache_load('sens')
if srows is None:
    srows=[]; t0=time.time()
    for name,ch in SENS:
        _apply(ch); Bq=blk_agg_w(Rhat)                              # model reward with the policy weight of the variant
        for sc in ['S0','S5']:
            for t in test_t:
                res=run_month(SCENARIOS[sc],Bq[:,t],B_all[:,t],B_press[:,t],B_nt[:,t])
                srows.append(dict(variant=name,scenario=sc,t=t,IA=res['IA'],IA_nt=res['IA_nt'],share=res['share_detected'],flight_h=res['flight_h']))
        print(f'  {name:16s} done  {(time.time()-t0)/60:.1f} min')
    _apply({}); cache_save('sens',srows)
_apply({})
# block size: the cells are aggregated again into smaller and larger blocks and the whole chain (reach, rewards, search times, routing, scoring) is rebuilt
BLKV=[0.8] if FAST_MODE else [0.4,0.8]; brows=cache_load('blocksize')
if brows is None:
    brows=[]; t0=time.time()
    try:
        for deg in BLKV:
            build_blocks(deg); kp=int(round(K_POOL*(BLK_DEG/deg)**2))
            for sc in ['S0','S5']:
                for t in test_t:
                    res=run_month(SCENARIOS[sc],B_rhat[:,t],B_all[:,t],B_press[:,t],B_nt[:,t],kpool=kp)
                    brows.append(dict(variant=f'blocks of {deg:g} deg',scenario=sc,t=t,IA=res['IA'],IA_nt=res['IA_nt'],share=res['share_detected'],flight_h=res['flight_h'],n_blocks=NB))
            print(f'  blocks of {deg:g} deg ({NB:,} blocks) done  {(time.time()-t0)/60:.1f} min',flush=True)
    finally: build_blocks(BLK_DEG)
    cache_save('blocksize',brows)
sens=pd.concat([pd.DataFrame(srows),pd.DataFrame(brows).drop(columns='n_blocks')],ignore_index=True); sens.to_csv(f'{OUT}/sensitivity_monthly.csv',index=False)
RESULTS['blocksize_n']={r_['variant']:int(r_['n_blocks']) for r_ in brows}
T6=[]
for name in [x_[0] for x_ in SENS]+[f'blocks of {deg:g} deg' for deg in BLKV]:
    a0=sens[(sens.variant==name)&(sens.scenario=='S0')].set_index('t'); a5=sens[(sens.variant==name)&(sens.scenario=='S5')].set_index('t')
    T6.append(dict(variant=name,IA_S0=a0.IA.mean(),IA_S5=a5.IA.mean(),gain_S5_pct=100*(a5.IA.mean()/a0.IA.mean()-1),ci_lo=boot_ratio_ci(a5.IA.values,a0.IA.values)[0],ci_hi=boot_ratio_ci(a5.IA.values,a0.IA.values)[1],
                   nt_capture_S0=a0.IA_nt.mean()/NT_REALIZED_PM,nt_capture_S5=a5.IA_nt.mean()/NT_REALIZED_PM,share_S0=a0.share.mean(),share_S5=a5.share.mean(),flight_h_S0=a0.flight_h.mean(),flight_h_S5=a5.flight_h.mean()))
T6=pd.DataFrame(T6); print(T6.round(3).to_string(index=False)); T6.to_csv(f'{OUT}/T6_sensitivity.csv',index=False); RESULTS['T6']=T6.round(5).to_dict('records')

## I. Ancho del anillo y jurisdicción

El backtest se repite restringiendo recompensas **y** puntaje a (i) anillos de 100 y 200 km y (ii) aguas a menos de 200 millas
de una costa. En cada variante se vuelve a elegir el despliegue con los datos del año de selección.

In [ ]:
import traceback
try:
    _BASE_B=dict(B_all=B_all,B_nt=B_nt,B_press=B_press,B_hist=B_hist,B_rhat=B_rhat,B_oracle=B_oracle,B_jur=B_jur)
    def masked_blocks(mask):
        mm=mask.astype(np.float64)[:,None]
        return dict(B_all=blk_agg(M_all*mm),B_nt=blk_agg(M_viol*mm),B_press=blk_agg(M_press*mm),B_hist=blk_agg_w(HIST*mm),B_rhat=blk_agg_w(Rhat*mm),B_oracle=blk_agg_w(M_all*mm),
                    B_jur=blk_agg(M_all*mm*JUR[:,None]) if JUR is not None else None)
    def series(scen,mt,ts_=None,col='IA',**kw):
        return np.array([evaluate(scen,mt,t,**kw)[col] for t in (test_t if ts_ is None else ts_)])
    def light(on):
        global K_POOL,SORTIE_SEEDS,SORTIE_SWAP
        if on: light._s=(K_POOL,SORTIE_SEEDS,SORTIE_SWAP); K_POOL,SORTIE_SEEDS,SORTIE_SWAP=150,1,False
        else: K_POOL,SORTIE_SEEDS,SORTIE_SWAP=light._s
    ring_d=cells.dist_bnd_km.values
    VARS=[('all rings (baseline)',np.ones(n,bool)),('rings <= 200 km',ring_d<=200),('rings <= 100 km',ring_d<=100)]
    if JUR is not None: VARS+=[('within 200 nm of a coast',JUR==1),('high seas only',JUR==0)]
    if FAST_MODE: VARS=VARS[:1]+VARS[2:4]
    TV=cache_load('variants')
    if TV is None:
        TV=[]; t0=time.time()
        for name,mask in VARS:
            globals().update(masked_blocks(mask))
            try:
                tot=float(B_all[:,test_t].sum(0).mean()); ntot=float(B_nt[:,test_t].sum(0).mean()); R_={}
                for sc in ['S0','S5']:
                    for mt in ['risk','hist','oracle']:
                        rs=[evaluate(SCENARIOS[sc],mt,t) for t in test_t]; R_[(sc,mt)]=np.array([r_['IA'] for r_ in rs]); R_[(sc,mt,'nt')]=np.array([r_['IA_nt'] for r_ in rs]); R_[(sc,mt,'w')]=np.array([r_['IA_w'] for r_ in rs])
                light(True)                                     # deployment selected within the variant (screening configuration, selection year)
                try:
                    sel=[np.mean([(lambda r_: r_['IA']+(W_NT-1)*r_['IA_nt'])(evaluate(d_,'hist',t)) for t in sel_t]) for d_ in DEPS]
                finally: light(False)
                iv=int(np.argmax(sel)); rk0=int((np.array(sel)>sel[i0]).sum())+1
                st=np.array([evaluate(DEPS[iv],'risk',t)['IA'] for t in test_t]); stn=np.array([evaluate(DEPS[iv],'risk',t)['IA_nt'] for t in test_t]) if ntot>0 else np.zeros(len(test_t))
                a0,a5,h0,h5=R_[('S0','risk')],R_[('S5','risk')],R_[('S0','hist')],R_[('S5','hist')]
                TV.append(dict(variant=name,cells=int(mask.sum()),realized_h=tot,IA_S0=a0.mean(),share_S0=a0.mean()/tot,IA_S5=a5.mean(),share_S5=a5.mean()/tot,
                    gain_S5=100*(a5.mean()/a0.mean()-1),gain_S5_lo=boot_ratio_ci(a5,a0)[0],gain_S5_hi=boot_ratio_ci(a5,a0)[1],
                    risk_vs_hist_S0=100*(a0.mean()/h0.mean()-1),rvh0_lo=boot_ratio_ci(a0,h0)[0],rvh0_hi=boot_ratio_ci(a0,h0)[1],
                    risk_vs_hist_S5=100*(a5.mean()/h5.mean()-1),rvh5_lo=boot_ratio_ci(a5,h5)[0],rvh5_hi=boot_ratio_ci(a5,h5)[1],
                    of_oracle_S0=R_[('S0','risk','w')].mean()/R_[('S0','oracle','w')].mean(),of_oracle_S5=R_[('S5','risk','w')].mean()/R_[('S5','oracle','w')].mean(),
                    nt_capture_S0=R_[('S0','risk','nt')].mean()/ntot if ntot>0 else np.nan,nt_capture_S5=R_[('S5','risk','nt')].mean()/ntot if ntot>0 else np.nan,
                    best=dep_name(DEPS[iv]),rank_S0_sel=rk0,IA_best=st.mean(),gain_best=100*(st.mean()/a0.mean()-1),gain_best_lo=boot_ratio_ci(st,a0)[0],gain_best_hi=boot_ratio_ci(st,a0)[1],
                    nt_capture_best=stn.mean()/ntot if ntot>0 else np.nan))
            finally: globals().update(_BASE_B)
            print(f'  {name:26s} done  {(time.time()-t0)/60:.1f} min',flush=True)
        cache_save('variants',TV)
    TV=pd.DataFrame(TV); print(TV.round(3).T.to_string()); TV.to_csv(f'{OUT}/T12_ring_jurisdiction.csv',index=False); RESULTS['T12']=TV.round(5).to_dict('records')
except Exception:
    RESULTS.setdefault('stage_errors',{})['I']=traceback.format_exc(); print('STAGE I FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['I'])

## K. Predictores alternativos, latencia de datos y métricas por bloque

In [ ]:
import traceback
try:
    ES=np.zeros_like(M_all)
    for t in range(1,NT): ES[:,t]=0.3*M_all[:,t-1]+0.7*ES[:,t-1]                 # exponential smoothing of the cell's past effort
    HIST_LAT=np.zeros_like(HIST); HIST_LAT[:,1:]=HIST[:,:-1]                       # historical map with data through t-2
    # risk model with data through t-2: dynamic features of the previous row of the same cell
    _c=cache_load('risk_lat')
    if _c is None:
        XL=X.copy(); j0_,j1_=len(STATIC),len(STATIC)+len(FEAT_DYN); XL[n:,j0_:j1_]=X[:-n,j0_:j1_]
        r_lat=np.zeros_like(y_all); t0=time.time()
        for rc,rname in REG.items():
            rm=region_idx==rc
            for T in TEST_YEARS:
                tr=rm&(year_idx<T-1); va=rm&(year_idx==T-1); te=rm&(year_idx==T)
                if tr.sum()==0 or te.sum()==0: continue
                m_=fit_model(XL[tr],y_all[tr],XL[va],y_all[va]); r_lat[te]=np.nan_to_num(np.maximum(m_.predict(XL[te]),0.0))
            print(f'  latency model {rname}: {(time.time()-t0)/60:.1f} min',flush=True)
        del XL; cache_save('risk_lat',r_lat)
    else: r_lat=_c
    Rlat=np.zeros((n,NT),np.float32); Rlat[cell_idx,t_idx]=r_lat
    PRED={'lag1':np.nan_to_num(F['lag1']),'roll3':np.nan_to_num(F['roll3']),'exp. smoothing':ES,'hist12, data to t-2':HIST_LAT,'risk model, data to t-2':Rlat}
    # cell-level ranking metrics by region (mean over test years)
    mrows=[]
    for name,Mx in list(PRED.items())+[('risk model',Rhat),('hist12',HIST)]:
        v=long(Mx)
        for rc,rname in REG.items():
            mm_=[monthly_metrics(y_all[(region_idx==rc)&(year_idx==T)],v[(region_idx==rc)&(year_idx==T)],t_idx[(region_idx==rc)&(year_idx==T)]) for T in TEST_YEARS]
            mrows.append(dict(predictor=name,region=rname,**{k_:float(np.mean([x_[k_] for x_ in mm_])) for k_ in ['spearman','top10_capture','auc_pos']}))
    T13a=pd.DataFrame(mrows); T13a.to_csv(f'{OUT}/T13a_predictors_cell.csv',index=False)
    # block-level ranking metrics (the planning unit), reachable blocks, all test months
    reach_any=np.zeros(NB,bool)
    for k_,v_ in REACH.items(): reach_any[v_]=True
    brow=[]
    for name,Mx in [('risk model',Rhat),('hist12',HIST)]+list(PRED.items()):
        Bx=blk_agg(Mx); sp_=[];cap_=[]
        for t in test_t:
            y_=B_all[reach_any,t]; p_=Bx[reach_any,t]
            if y_.sum()<=0: continue
            sp_.append(stats.spearmanr(y_,p_).correlation); k_=max(1,int(0.1*len(y_))); cap_.append(y_[np.argsort(-p_)[:k_]].sum()/y_.sum())
        brow.append(dict(predictor=name,spearman_block=float(np.nanmean(sp_)),top10_capture_block=float(np.mean(cap_))))
    T13b=pd.DataFrame(brow); T13b.to_csv(f'{OUT}/T13b_predictors_block.csv',index=False)
    # decision-level comparison: the risk model against each alternative reward
    _c=cache_load('pred_bt')
    if _c is None:
        _c={}
        for name,Mx in PRED.items():
            Bx=blk_agg_w(Mx)
            for sc in ['S0','S5']: _c[(name,sc)]=np.array([evaluate(SCENARIOS[sc],'x',t,reward=Bx[:,t])['IA'] for t in test_t])
            print(f'  {name}: done',flush=True)
        cache_save('pred_bt',_c)
    prow=[]
    for name in PRED:
        d=dict(predictor=name)
        for sc in ['S0','S5']:
            a=ser(sc,'risk').values; b=_c[(name,sc)]; lo,hi=boot_ratio_ci(a,b)
            d.update({f'IA_{sc}':b.mean(),f'risk_gain_{sc}':100*(a.mean()/b.mean()-1),f'lo_{sc}':lo,f'hi_{sc}':hi,f'wins_{sc}':int((a>b).sum())})
        prow.append(d)
    T13=pd.DataFrame(prow).merge(T13b,on='predictor',how='left'); print(T13.round(3).to_string(index=False)); print(T13b.round(3).to_string(index=False))
    T13.to_csv(f'{OUT}/T13_predictors.csv',index=False); RESULTS['T13']=T13.round(5).to_dict('records'); RESULTS['T13a']=T13a.round(4).to_dict('records'); RESULTS['T13b']=T13b.round(4).to_dict('records')
except Exception:
    RESULTS.setdefault('stage_errors',{})['K']=traceback.format_exc(); print('STAGE K FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['K'])

## L. Certificación ampliada: cotas con otras recompensas y escenarios, sensibilidad de la cota, y solución exacta de instancias reducidas

In [ ]:
import traceback
try:
    SCENARIOS['S*']=DEPS[i_star]; SC_LABEL['S*']='S* selected'
    LP2_T=test_t[::4] if FAST_MODE else test_t[::2]
    _c=cache_load('lp_ext')
    if _c is None:
        _c=[]; t0=time.time()
        for sc in ['S0','S5','S6','S*']:
            for mt in ['oracle','risk','hist']:
                for t in LP2_T:
                    rho_=reward_vec(mt,t); _c.append(dict(scenario=sc,reward=mt,config='full',t=t,planned=evaluate(SCENARIOS[sc],mt,t)['planned'],UB=lp_upper_bound(SCENARIOS[sc],rho_)))
            print(f'  LP bounds {sc}: {(time.time()-t0)/60:.1f} min',flush=True)
        light(True)
        try:
            for sc in ['S0','S5']:
                for t in LP2_T: _c.append(dict(scenario=sc,reward='oracle',config='screening',t=t,planned=evaluate(SCENARIOS[sc],'oracle',t)['planned'],UB=lp_upper_bound(SCENARIOS[sc],reward_vec('oracle',t))))
        finally: light(False)
        for L_ in ([4,8] if FAST_MODE else [4,8,10]):                 # sensitivity of the bound to the number of radial levels
            for t in LP2_T[::2]: _c.append(dict(scenario='S5',reward='oracle',config=f'levels={L_}',t=t,planned=np.nan,UB=lp_upper_bound(SCENARIOS['S5'],reward_vec('oracle',t),L=L_)))
        cache_save('lp_ext',_c)
    _c=[r_ for r_ in _c if r_['config']!='q=0.5']
    _c5=cache_load('lp_q05')                                            # gap when the reward saturates more slowly (q = 0.5)
    if _c5 is None:
        _c5=[]; _q=PRESENCE; PRESENCE=0.5
        try:
            for sc in ['S0','S5']:
                for t in LP2_T[::2]: _c5.append(dict(scenario=sc,reward='oracle',config='q=0.5',t=t,planned=evaluate(SCENARIOS[sc],'oracle',t)['planned'],UB=lp_upper_bound(SCENARIOS[sc],reward_vec('oracle',t))))
        finally: PRESENCE=_q
        cache_save('lp_q05',_c5)
    lpx=pd.DataFrame(_c+_c5); lpx.to_csv(f'{OUT}/lp_bound_extended.csv',index=False)
    g_=lpx[lpx.planned.notna()]; assert (g_.planned<=g_.UB*(1+1e-6)).all(), 'a plan exceeds its upper bound'
    T14=g_.groupby(['config','scenario','reward']).apply(lambda d: pd.Series(dict(months=len(d),of_UB=d.planned.sum()/d.UB.sum(),of_UB_min=(d.planned/d.UB).min()))).reset_index()
    _ref=lpx[(lpx.scenario=='S5')&(lpx.reward=='oracle')&(lpx.config=='full')].set_index('t').UB
    T14L=pd.DataFrame([dict(levels=int(c_.split('=')[1]),UB_rel=float((d.set_index('t').UB/_ref.reindex(d.t.values).values).mean())) for c_,d in lpx[lpx.config.str.startswith('levels')].groupby('config')])
    print(T14.round(4).to_string(index=False)); print(T14L.round(4).to_string(index=False)); T14.to_csv(f'{OUT}/T14_lp_extended.csv',index=False)
    RESULTS['T14']=T14.round(5).to_dict('records'); RESULTS['T14_levels']=T14L.round(5).to_dict('records')
    # bounds with uniform rewards, and the role of each of the two time constraints of the relaxation
    _c=cache_load('lp_ext2')
    if _c is None:
        _c=dict(uniform=[],ablation=[]); t0=time.time()
        for sc in ['S0','S5']:
            for t in LP2_T[::2]:
                rho_=reward_vec('uniform',t); _c['uniform'].append(dict(scenario=sc,t=t,planned=evaluate(SCENARIOS[sc],'uniform',t)['planned'],UB=lp_upper_bound(SCENARIOS[sc],rho_)))
        for t in LP2_T[::2]:
            rho_=reward_vec('oracle',t); sc_=SCENARIOS['S5']
            _c['ablation'].append(dict(t=t,both=lp_upper_bound(sc_,rho_),far_only=lp_upper_bound(sc_,rho_,near_c=False),near_only=lp_upper_bound(sc_,rho_,far_c=False)))
        print(f'  LP bounds, uniform rewards and ablation: {(time.time()-t0)/60:.1f} min',flush=True); cache_save('lp_ext2',_c)
    _u=pd.DataFrame(_c['uniform']); _a=pd.DataFrame(_c['ablation']); assert (_u.planned<=_u.UB*(1+1e-6)).all()
    RESULTS['T14_uniform']=[dict(scenario=sc,months=int(len(d_)),of_UB=float(d_.planned.sum()/d_.UB.sum()),of_UB_min=float((d_.planned/d_.UB).min())) for sc,d_ in _u.groupby('scenario')]
    RESULTS['lp_ablation']=dict(far_only_rel=float((_a.far_only/_a.both).mean()),near_only_rel=float((_a.near_only/_a.both).mean()),months=int(len(_a)))
    print(RESULTS['T14_uniform'],RESULTS['lp_ablation'])
except Exception:
    RESULTS.setdefault('stage_errors',{})['L']=traceback.format_exc(); print('STAGE L FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['L'])

In [ ]:
import traceback
try:
    # ---------------- exact solutions of reduced instances ----------------
    # The RB-SRP is solved to proven optimality on instances with 12 and 16 blocks through an equivalent route-based formulation:
    # every feasible route of every aircraft group is enumerated by dynamic programming over subsets (only inclusion-maximal block sets
    # are kept, since the objective is monotone), and an integer program chooses how many times each route is flown.
    from scipy.optimize import milp, LinearConstraint, Bounds
    def maximal_routes(base,plat,B):
        """Inclusion-maximal sets of blocks of B that one sortie of (base, plat) can search within the time available."""
        m=len(B); v=PL[plat]['v']; Tb=tb_h(plat); tn=tau_h(plat)[B]; d0=DBASE[base][B]/v; Dn=DKM[np.ix_(B,B)].astype(float)/v
        full=1<<m; dp=np.full((full,m),np.inf); feas=np.zeros(full,bool)
        for j in range(m):
            if 2*d0[j]+tn[j]<=Tb+1e-9: dp[1<<j,j]=d0[j]+tn[j]
        inS=((np.arange(full)[:,None]>>np.arange(m)[None,:])&1).astype(bool)
        for Sx in range(1,full):
            row=dp[Sx]
            if not np.isfinite(row).any(): continue
            feas[Sx]=True; ext=(row[:,None]+Dn+tn[None,:]).min(0)
            for k in np.where(~inS[Sx]&(ext+d0<=Tb+1e-9))[0]:
                S2=Sx|(1<<k)
                if ext[k]<dp[S2,k]: dp[S2,k]=ext[k]
        idx=np.where(feas)[0]; keep=[]
        for Sx in idx:
            if not any(feas[Sx|(1<<k)] for k in range(m) if not (Sx>>k)&1): keep.append(Sx)
        return [[B[k] for k in range(m) if (Sx>>k)&1] for Sx in keep]

    def exact_small(scen,rho):
        """Optimal value of the RB-SRP for deployment `scen` when only the blocks with rho>0 carry reward."""
        DET=PMIN*PRESENCE; pos=[int(b) for b in np.where(rho>0)[0]]; bi={b:i for i,b in enumerate(pos)}; groups,cap=month_slots(scen)
        cols=[]; gi=[]
        for g_i,g in enumerate(groups):
            B=[b for b in pos if b in set(g['ok'].tolist())]
            for r_ in (maximal_routes(g['base'],g['plat'],B) if B else []): cols.append(r_); gi.append(g_i)
        nS=sum(min(g['left'],cap[g['base']]) for g in groups); nr=len(cols); nb=len(pos); nv=nr+nb*nS
        c=np.zeros(nv)
        for i,b in enumerate(pos):
            for k in range(nS): c[nr+i*nS+k]=-rho[b]*DET*(1-DET)**k
        A=sp.lil_matrix((nb+len(groups),nv)); lo=np.zeros(nb+len(groups)); hi=np.zeros(nb+len(groups))
        for j,r_ in enumerate(cols):
            for b in r_: A[bi[b],j]=-1.0
            A[nb+gi[j],j]=1.0
        for i in range(nb): A[i,nr+i*nS:nr+(i+1)*nS]=1.0
        for g_i,g in enumerate(groups): lo[nb+g_i]=0.0; hi[nb+g_i]=min(g['left'],cap[g['base']])
        ub=np.concatenate([np.full(nr,float(nS)),np.ones(nb*nS)])
        res=milp(c,constraints=LinearConstraint(A.tocsr(),lo,hi),integrality=np.ones(nv),bounds=Bounds(np.zeros(nv),ub),options=dict(time_limit=600,mip_rel_gap=1e-9))
        return (-res.fun if res.x is not None else np.nan), int(res.status), nr

    def small_instance(rng,m_blocks,n_sorties):
        """A reduced instance drawn from the backtest: one or two aircraft groups, a few sorties, and the most valuable blocks within their reach."""
        while True:
            t=int(rng.choice(test_t)); gs=[(b,p) for b,ac in SCENARIOS['S6'].items() for p in ac]; k=int(rng.integers(1,3))
            pick=[gs[i] for i in rng.choice(len(gs),k,replace=False)]; scen={}
            for b,p in pick: scen.setdefault(b,{})[p]=1
            ok=np.unique(np.concatenate([REACH[g] for g in pick])); rho=np.zeros(NB); top=ok[np.argsort(-B_oracle[ok,t],kind='stable')[:m_blocks]]
            if len(top)<m_blocks or B_oracle[top,t].min()<=0: continue
            rho[top]=B_oracle[top,t]; return scen,rho,max(1,n_sorties//k)
    EX=cache_load('exact')
    if EX is None:
        EX=[]; rng=np.random.default_rng(SEED+5); _n=SORTIES_PER_AC_MONTH; _cap=BASES.sorties_wk.copy(); t0=time.time()
        for size,(mb,ns,ninst) in {'12 blocks':(12,4,6 if FAST_MODE else 40),'16 blocks':(16,8,2 if FAST_MODE else 20)}.items():
            for _ in range(ninst):
                scen,rho,per=small_instance(rng,mb,ns)
                SORTIES_PER_AC_MONTH=per; BASES['sorties_wk']=99
                try:
                    h=run_month(scen,rho,rho,rho,np.zeros(NB))['planned']; ub=lp_upper_bound(scen,rho); opt,stt,nr=exact_small(scen,rho)
                    EX.append(dict(size=size,blocks=mb,sorties=len(slot_order(scen)),groups=len(month_slots(scen)[0]),routes=nr,heur=h,opt=opt,lp=ub,status=stt))
                finally: SORTIES_PER_AC_MONTH=_n; BASES['sorties_wk']=_cap
            print(f'  exact, {size}: {sum(e_["size"]==size for e_ in EX)} instances, {(time.time()-t0)/60:.1f} min',flush=True)
        cache_save('exact',EX)
    EXD=pd.DataFrame(EX); EXD=EXD[EXD.status==0]; EXD['heur_of_opt']=EXD.heur/EXD.opt; EXD['lp_over_opt']=EXD.lp/EXD.opt; EXD.to_csv(f'{OUT}/exact_instances.csv',index=False)
    assert (EXD.heur<=EXD.opt*(1+1e-6)).all() and (EXD.lp>=EXD.opt*(1-1e-6)).all(), 'ordering heuristic <= optimum <= LP bound violated'
    T15=EXD.groupby('size').agg(instances=('opt','size'),sorties=('sorties','mean'),routes=('routes','mean'),heur_mean=('heur_of_opt','mean'),heur_min=('heur_of_opt','min'),heur_optimal_pct=('heur_of_opt',lambda s_:100*(s_>1-1e-6).mean()),
                                lp_over_opt_mean=('lp_over_opt','mean'),lp_over_opt_max=('lp_over_opt','max')).reset_index()
    print(T15.round(4).to_string(index=False)); T15.to_csv(f'{OUT}/T15_exact.csv',index=False); RESULTS['T15']=T15.round(5).to_dict('records')
    # ---- a certified lower bound on the quality of single sorties planned over the full candidate list (first sortie of each group and month)
    _c=cache_load('single_lp')
    if _c is None:
        _c=[]; _n=SORTIES_PER_AC_MONTH; _cap=BASES.sorties_wk.copy(); SORTIES_PER_AC_MONTH=1; BASES['sorties_wk']=99
        try:
            for t in (test_t[::6] if FAST_MODE else test_t[::2]):
                for b,ac in SCENARIOS['S6'].items():
                    for p in ac:
                        for mt in ['risk','oracle']:
                            rho=reward_vec(mt,t); sc_={b:{p:1}}; h=run_month(sc_,rho,rho,rho,np.zeros(NB))['planned']
                            if h>0: _c.append(dict(t=t,base=b,plat=p,reward=mt,heur=h,lp=lp_upper_bound(sc_,rho)))
        finally: SORTIES_PER_AC_MONTH=_n; BASES['sorties_wk']=_cap
        cache_save('single_lp',_c)
    sl=pd.DataFrame(_c); sl['ratio']=sl.heur/sl.lp
    T15b=sl.groupby(['reward','plat']).agg(instances=('ratio','size'),mean=('ratio','mean'),p10=('ratio',lambda s_:s_.quantile(0.1)),min=('ratio','min')).reset_index()
    print(T15b.round(3).to_string(index=False)); RESULTS['T15b']=T15b.round(5).to_dict('records'); sl.to_csv(f'{OUT}/single_sortie_lp.csv',index=False)
except Exception:
    RESULTS.setdefault('stage_errors',{})['L']=traceback.format_exc(); print('STAGE L FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['L'])

## L2. Salida individual resuelta de forma exacta sobre la lista de candidatos
Ramificación y corte (Gurobi) sobre el modelo de ruteo selectivo de una salida, con cortes de conectividad separados en soluciones enteras y fraccionarias. Con la licencia restringida que trae `gurobipy` se resuelven listas de 40 y 60 candidatos; si en tu Drive hay un `gurobi.lic` válido para Colab (o indicas su ruta en `GUROBI_LIC`), se resuelve además la lista completa de candidatos. Si Gurobi no está disponible, la etapa se omite y el resto del cuaderno sigue.

In [ ]:
import glob, sys, traceback
from scipy.sparse.csgraph import connected_components
GUROBI_LIC = ''        # optional: full path of a gurobi.lic valid on this machine (e.g. a WLS license); empty = search the Drive
def exact_op(D,tau,r,Tb,env,start=None,time_limit=120):
    """Optimal single sortie by branch and cut. Undirected selective-routing model: y_j block searched, x_e leg flown, x0_j legs between
    base and block j (2 = out and back); degree, time and connectivity constraints, the latter separated in the callback.
    Node 0 = base. Returns (value, upper bound, proven optimal, seconds)."""
    import gurobipy as gp; from gurobipy import GRB
    t0=time.time(); ok=np.where((r[1:]>0)&(2*D[0,1:]+tau[1:]<=Tb+1e-9))[0]+1; n=len(ok)
    if n==0: return 0.0,0.0,True,0.0
    Dn=D[np.ix_(ok,ok)]; d0=D[0,ok]; tn=tau[ok]; rn=r[ok]
    iu,ju=np.triu_indices(n,1); fe=d0[iu]+tn[iu]+Dn[iu,ju]+tn[ju]+d0[ju]<=Tb+1e-9; iu=iu[fe]; ju=ju[fe]; ne=len(iu)
    M=gp.Model(env=env); M.Params.LazyConstraints=1; M.Params.TimeLimit=time_limit; M.Params.MIPGap=1e-6
    y=M.addMVar(n,vtype=GRB.BINARY,obj=-rn); x0=M.addMVar(n,vtype=GRB.INTEGER,lb=0,ub=2); xe=M.addMVar(ne,vtype=GRB.BINARY)
    Inc=sp.csr_matrix((np.ones(2*ne),(np.concatenate([iu,ju]),np.concatenate([np.arange(ne),np.arange(ne)]))),shape=(n,ne))
    M.addConstr(Inc@xe+x0-2*y==0); M.addConstr(x0.sum()==2); M.addConstr(tn@y+d0@x0+Dn[iu,ju]@xe<=Tb); M.update()
    yv=y.tolist(); x0v=x0.tolist(); xev=xe.tolist()
    def components(xs,ys,thr):
        k=np.where(xs>thr)[0]; g=sp.csr_matrix((np.ones(len(k)),(iu[k],ju[k])),shape=(n,n))
        _,lab=connected_components(g,directed=False); act=np.where(ys>1e-6)[0]
        return [act[lab[act]==c] for c in np.unique(lab[act])]
    def cut(model,S,ys):
        inS=np.zeros(n,bool); inS[S]=True; k=np.where(inS[iu]!=inS[ju])[0]; i=int(S[np.argmax(ys[S])])
        model.cbLazy(gp.quicksum([xev[q] for q in k])+gp.quicksum([x0v[j] for j in S])>=2*yv[i])
    def cb(model,where):
        if where==GRB.Callback.MIPSOL:
            xs=np.array(model.cbGetSolution(xev)); ys=np.array(model.cbGetSolution(yv)); zs=np.array(model.cbGetSolution(x0v))
            for S in components(xs,ys,0.5):
                if zs[S].sum()<0.5: cut(model,S,ys)
        elif where==GRB.Callback.MIPNODE and model.cbGet(GRB.Callback.MIPNODE_STATUS)==GRB.OPTIMAL:
            xs=np.array(model.cbGetNodeRel(xev)); ys=np.array(model.cbGetNodeRel(yv)); zs=np.array(model.cbGetNodeRel(x0v))
            for S in components(xs,ys,1e-6):
                if len(S)>1 and zs[S].sum()<1e-6: cut(model,S,ys)
    if start is not None:
        pos={int(b_):i for i,b_ in enumerate(ok)}
        for v_ in yv: v_.Start=0
        for b_ in start:
            if int(b_) in pos: yv[pos[int(b_)]].Start=1
    M.optimize(cb)
    val=-M.ObjVal if M.SolCount else np.nan; bd=-M.ObjBound; pr=M.Status==GRB.OPTIMAL; M.dispose()
    return float(val),float(bd),bool(pr),time.time()-t0

def gurobi_env():
    """Gurobi environment and whether the license allows models of any size."""
    import gurobipy as gp
    cand=[GUROBI_LIC] if GUROBI_LIC else []
    try:                                                   # WLS license kept in Colab Secrets (WLSACCESSID, WLSSECRET, LICENSEID)
        from google.colab import userdata
        acc,sec,lid=userdata.get('WLSACCESSID'),userdata.get('WLSSECRET'),userdata.get('LICENSEID')
        if acc and sec and lid:
            open('/content/gurobi.lic','w').write(f"WLSACCESSID={acc}\nWLSSECRET={sec}\nLICENSEID={lid}\n"); cand.append('/content/gurobi.lic'); print('Gurobi license written from Colab Secrets')
    except Exception as ex: print('Colab Secrets not used for the Gurobi license:',type(ex).__name__)
    cand+=[f'{IN}/gurobi.lic']
    for pat in ['','*/','*/*/','*/*/*/']: cand+=glob.glob(f'/content/drive/MyDrive/{pat}gurobi.lic')
    cand=[c for c in cand if c and os.path.exists(c)]
    def start():
        e=gp.Env(empty=True); e.setParam('OutputFlag',0); e.start(); return e
    env=None
    for c in cand:
        try: os.environ['GRB_LICENSE_FILE']=c; env=start(); print('Gurobi license file:',c); break
        except Exception as ex: print('license not usable here:',c,'|',str(ex)[:120]); os.environ.pop('GRB_LICENSE_FILE',None)
    if env is None: env=start()
    try:
        m=gp.Model(env=env); m.addMVar(2500,ub=1.0); m.optimize(); full=True
    except gp.GurobiError: full=False
    return env,full

def op_instance(rng):
    """One sortie as the planner meets it: a random test month and aircraft group, with the predicted reward of the month or, half of the
    time, the reward left after the whole month has been planned (what later sorties and the re-optimization pass face)."""
    while True:
        t=int(rng.choice(test_t)); gs=[(b,p) for b,ac in SCENARIOS['S6'].items() for p in ac]; base,plat=gs[int(rng.integers(len(gs)))]
        rho=B_rhat[:,t].astype(np.float64); resid=bool(rng.random()<0.5)
        if resid: rho=rho*(1-PMIN*PRESENCE)**run_month(SCENARIOS['S6'],rho,rho,rho,np.zeros(NB))['nvis']
        ok=REACH[(base,plat)]; order=ok[np.argsort(-rho[ok],kind='stable')]; order=order[rho[order]>0]
        if len(order)>=60: return dict(t=t,base=base,plat=plat,resid=resid),rho,order
def op_matrices(base,plat,rho,pool):
    v=PL[plat]['v']; m=len(pool)+1; D=np.zeros((m,m)); D[1:,1:]=DKM[np.ix_(pool,pool)]; D[0,1:]=D[1:,0]=DBASE[base][pool]; D/=v
    return D,np.concatenate([[0.0],tau_h(plat)[pool]]),np.concatenate([[0.0],rho[pool]]),tb_h(plat)
try:
    try: import gurobipy
    except ImportError:
        import subprocess; subprocess.run([sys.executable,'-m','pip','-q','install','gurobipy'],check=False); import gurobipy
    GENV,GRB_FULL=gurobi_env(); print('Gurobi',gurobipy.gurobi.version(),'| full license:',GRB_FULL)
    OPX=cache_load('exact_op'); n_inst=4 if FAST_MODE else 24; n_full=(1 if FAST_MODE else 8) if GRB_FULL else 0
    need_small=OPX is None; need_full=n_full>0 and not any(r_['L']>60 for r_ in (OPX or []))
    if need_small or need_full:            # the instances are drawn again with the same seed, so a later run with a full license only adds the full lists
        LIM={40:60,60:180,K_POOL:450}; rng=np.random.default_rng(SEED+6); OPX=OPX or []; t0=time.time()
        for i_ in range(n_inst if need_small else n_full):
            meta,rho,order=op_instance(rng)
            for Lc in ([40,60] if need_small else [])+([K_POOL] if (need_full and i_<n_full) else []):
                pool=order[:Lc]; D,tau_,r_,Tb=op_matrices(meta['base'],meta['plat'],rho,pool)
                rt,hrs,h=solve_sortie(D,tau_,r_,Tb); val,bd,pr,sec=exact_op(D,tau_,r_,Tb,GENV,start=rt,time_limit=LIM[Lc])
                OPX.append(dict(meta,inst=i_,L=int(Lc),n=len(pool),blocks=len(rt),heur=h,opt=max(val,h) if val==val else h,bound=bd,proven=pr,sec=sec))
            if (i_+1)%4==0 or not need_small: print(f'  exact sorties: {i_+1}/{n_inst if need_small else n_full}  ({(time.time()-t0)/60:.1f} min)',flush=True)
        cache_save('exact_op',OPX)
    OPD=pd.DataFrame(OPX); OPD['heur_of_opt']=OPD.heur/OPD.opt; OPD['heur_of_bound']=OPD.heur/OPD.bound; OPD.to_csv(f'{OUT}/exact_sortie_candidates.csv',index=False)
    assert (OPD.heur<=OPD.bound*(1+1e-6)).all(), 'heuristic above the certified bound'
    T15c=OPD.groupby('L').agg(instances=('opt','size'),proven=('proven','sum'),blocks=('blocks','mean'),heur_mean=('heur_of_opt','mean'),heur_min=('heur_of_opt','min'),
                              heur_optimal_pct=('heur_of_opt',lambda s_:100*(s_>1-1e-6).mean()),certified_min=('heur_of_bound','min'),sec_mean=('sec','mean'),sec_max=('sec','max')).reset_index()
    w_=OPD.pivot(index='inst',columns='L',values='opt')
    RESULTS['T15c']=T15c.round(5).to_dict('records'); RESULTS['exact_op_list_growth']={f'{a}->{b}':float((w_[b]/w_[a]).dropna().mean()-1) for a,b in zip(w_.columns[:-1],w_.columns[1:])}
    print(T15c.round(4).to_string(index=False)); print('mean increase of the optimum when the candidate list grows:',RESULTS['exact_op_list_growth'])
except Exception:
    RESULTS['stage_L2_error']=traceback.format_exc(); print('STAGE L2 SKIPPED (the rest of the notebook continues):'); print(RESULTS['stage_L2_error'])

## M. Basamiento: indicador simple de cobertura, selección con origen rodante, destacamentos estacionales y S* con la configuración completa

In [ ]:
import traceback
try:
    # S* and the reference scenarios with the full configuration
    SF={}
    for mt in ['risk','hist','oracle']:
        rs=[evaluate(SCENARIOS['S*'],mt,t) for t in test_t]; SF[mt]=np.array([r_['IA'] for r_ in rs]); SF[mt+'_nt']=np.array([r_['IA_nt'] for r_ in rs]); SF[mt+'_w']=np.array([r_['IA_w'] for r_ in rs]); SF[mt+'_jur']=np.array([r_['IA_jur'] for r_ in rs]); SF[mt+'_fh']=np.array([r_['flight_h'] for r_ in rs])
    a0=ser('S0','risk').values; a5=ser('S5','risk').values
    STARF=dict(name=dep_name(SCENARIOS['S*']),IA=float(SF['risk'].mean()),share=float(SF['risk'].mean()/RESULTS['all_realized_per_month_test']),nt_capture=float(SF['risk_nt'].mean()/NT_REALIZED_PM),IA_jur=float(np.nanmean(SF['risk_jur'])),
               flight_h=float(SF['risk_fh'].mean()),of_oracle=float(SF['risk_w'].mean()/SF['oracle_w'].mean()),delta_vs_S0=float(100*(SF['risk'].mean()/a0.mean()-1)),ci_vs_S0=boot_ratio_ci(SF['risk'],a0),
               delta_vs_S5=float(100*(SF['risk'].mean()/a5.mean()-1)),ci_vs_S5=boot_ratio_ci(SF['risk'],a5),risk_vs_hist=float(100*(SF['risk'].mean()/SF['hist'].mean()-1)),ci_risk_vs_hist=boot_ratio_ci(SF['risk'],SF['hist']))
    RESULTS['star_full']=STARF; print(json.dumps(STARF,indent=1,default=str))
    # a heavier search for the final comparisons: more seeded starts per sortie and more re-optimization passes
    _c=cache_load('intensive')
    if _c is None:
        _c={}; _keep=(REOPT_PASSES,SORTIE_SEEDS); REOPT_PASSES,SORTIE_SEEDS=4,8; t0=time.time()
        try:
            for sc in ['S0','S5','S*']:
                for mt in ['risk','hist']: _c[(sc,mt)]=np.array([evaluate(SCENARIOS[sc],mt,t)['IA'] for t in (test_t[::4] if FAST_MODE else test_t)])
        finally: REOPT_PASSES,SORTIE_SEEDS=_keep
        print(f'  heavier search: {(time.time()-t0)/60:.1f} min',flush=True); cache_save('intensive',_c)
    if not FAST_MODE:
        _b={('S0','risk'):a0,('S5','risk'):a5,('S0','hist'):ser('S0','hist').values,('S5','hist'):ser('S5','hist').values,('S*','risk'):SF['risk'],('S*','hist'):SF['hist']}
        INT=dict(change_pct={f'{k_[0]}|{k_[1]}':float(100*(_c[k_].mean()/_b[k_].mean()-1)) for k_ in _c})
        for lab,x_,y_ in [('risk_vs_hist_S0',('S0','risk'),('S0','hist')),('risk_vs_hist_S5',('S5','risk'),('S5','hist')),('S5_vs_S0',('S5','risk'),('S0','risk')),('Sstar_vs_S0',('S*','risk'),('S0','risk')),('Sstar_vs_S5',('S*','risk'),('S5','risk'))]:
            INT[lab]=dict(base=float(100*(_b[x_].mean()/_b[y_].mean()-1)),heavy=float(100*(_c[x_].mean()/_c[y_].mean()-1)),heavy_ci=boot_ratio_ci(_c[x_],_c[y_]))
        RESULTS['intensive']=INT; print(json.dumps(INT,indent=1,default=str))
    # coverage indicator: realized effort of the period within reach of at least one aircraft of the deployment (no routing)
    def covered(dep,ts_):
        m_=np.zeros(NB,bool)
        for b,ac in dep.items():
            for p in ac: m_[REACH[(b,p)]]=True
        return float(B_all[m_][:,ts_].sum(1).sum()/len(ts_))
    en['cov_sel']=[covered(d_,sel_t) for d_ in DEPS]; en['cov_test']=[covered(d_,test_t) for d_ in DEPS]
    e_=en[en.IA_test.notna()]; topc=e_[e_.cov_sel>=e_.cov_sel.max()*0.999]
    COV=dict(spearman_cov_IA_sel=float(stats.spearmanr(en.cov_sel,en.IA).correlation),spearman_cov_IA_test=float(stats.spearmanr(e_.cov_test,e_.IA_test).correlation),
             n_max_coverage=int(len(topc)),IA_test_min_maxcov=float(topc.IA_test.min()),IA_test_max_maxcov=float(topc.IA_test.max()),rank_test_best_maxcov=int(topc.rank_test.min()),rank_test_worst_maxcov=int(topc.rank_test.max()),
             nt_capture_min_maxcov=float(topc.nt_capture_test.min()),nt_capture_max_maxcov=float(topc.nt_capture_test.max()),IA_test_range_all=[float(e_.IA_test.min()),float(e_.IA_test.max())])
    # selection with a rolling origin: choose with year Y-1, evaluate in year Y (screening configuration, risk-based planning)
    yy=np.array([Y0+t//12 for t in test_t]); ROLL=[]
    if not FAST_MODE:
        for Y in sorted(set(yy))[1:]:
            s_=OOS_IA[:,yy==Y-1].mean(1)+(W_NT-1)*OOS_NT[:,yy==Y-1].mean(1); ev=OOS_IA[:,yy==Y].mean(1); k_=int(np.argmax(s_))
            ROLL.append(dict(select_on=int(Y-1),evaluate_in=int(Y),selected=dep_name(DEPS[k_]),rank=int((ev>ev[k_]).sum())+1,regret_pct=float(100*(1-ev[k_]/ev.max())),gain_vs_S0=float(100*(ev[k_]/ev[i0]-1)),
                             rank_Sstar=int((ev>ev[i_star]).sum())+1,rank_S0=int((ev>ev[i0]).sum())+1,nt_capture=float(OOS_NT[k_,yy==Y].mean()/max(B_nt[:,[t for t in test_t if Y0+t//12==Y]].sum(0).mean(),1e-9))))
        # seasonal detachments: for each quarter, the deployment that was best in the same quarter of the previous year
        qq=np.array([(t%12)//3 for t in test_t]); fixed=[]; seas=[]
        for Y in sorted(set(yy))[1:]:
            kf=int(np.argmax(OOS_IA[:,yy==Y-1].mean(1)))
            for q_ in range(4):
                cur=(yy==Y)&(qq==q_); prev=(yy==Y-1)&(qq==q_); ks=int(np.argmax(OOS_IA[:,prev].mean(1))); fixed+=list(OOS_IA[kf,cur]); seas+=list(OOS_IA[ks,cur])
        COV['seasonal_gain_pct']=float(100*(np.mean(seas)/np.mean(fixed)-1)); COV['seasonal_ci']=boot_ratio_ci(np.array(seas),np.array(fixed))
    RESULTS['coverage_indicator']=COV; RESULTS['rolling_selection']=ROLL; print(json.dumps(COV,indent=1,default=str)); print(pd.DataFrame(ROLL).round(2).to_string(index=False))
    en.to_csv(f'{OUT}/basing_enumeration.csv',index=False)
    # robustness of the intervals to the bootstrap block length
    BB=[]
    for lab,a,b in [('risk vs historical, S0',a0,ser('S0','hist').values),('risk vs historical, S5',a5,ser('S5','hist').values),('S5 vs S0',a5,a0),('S* vs S0',SF['risk'],a0),('S* vs S5',SF['risk'],a5)]:
        BB.append(dict(comparison=lab,gain=100*(a.mean()/b.mean()-1),**{f'b{k_}_{s_}':boot_ratio_ci(a,b,block=k_)[i_] for k_ in (3,6,12) for i_,s_ in enumerate(('lo','hi'))}))
    RESULTS['bootstrap_blocks']=BB; print(pd.DataFrame(BB).round(1).to_string(index=False))
except Exception:
    RESULTS.setdefault('stage_errors',{})['M']=traceback.format_exc(); print('STAGE M FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['M'])

## N. Datos diarios por buque, SAR y detalle no-take
Requiere las salidas del Notebook 1b en `acanav_paper1/addon/`. Si no están, la etapa se omite.

In [ ]:
import traceback
try:
    AD=f'{IN}/addon'; HAS_DAILY=os.path.exists(f'{AD}/daily_block_vessel.parquet'); print('daily data:',HAS_DAILY)
    if HAS_DAILY:
        dv=pd.read_parquet(f'{AD}/daily_block_vessel.parquet'); dv['date']=pd.to_datetime(dv.date)
        dv=dv.merge(blocks[['blk_lon','blk_lat','blk_id']].round({'blk_lon':4,'blk_lat':4}),on=['blk_lon','blk_lat'],how='inner')
        dv['t']=(dv.date.dt.year-Y0)*12+dv.date.dt.month-1; dv['day']=dv.date.dt.day; dv['dim']=dv.date.dt.days_in_month
        dvt=dv[dv.t.isin(test_t)]
        # ---- persistence of the activity within the month (test period)
        bm=dvt.groupby(['blk_id','t']).agg(hours=('hours','sum'),days=('day','nunique'),dim=('dim','first'),vessels=('vessel','nunique')).reset_index(); bm['presence']=bm.days/bm.dim
        vb=dvt.groupby(['blk_id','t','vessel']).agg(hours=('hours','sum'),days=('day','nunique')).reset_index()
        PERS=dict(block_months=int(len(bm)),presence_mean=float(bm.presence.mean()),presence_effort_weighted=float(np.average(bm.presence,weights=bm.hours)),
                  active_days_effort_weighted=float(np.average(bm.days,weights=bm.hours)),presence_quartiles_effort_weighted=[float(bm.sort_values('presence').presence.iloc[int(np.searchsorted(bm.sort_values('presence').hours.cumsum().values,q_*bm.hours.sum()))]) for q_ in (0.25,0.5,0.75)],
                  vessel_days_in_block_mean=float(vb.days.mean()),vessel_days_in_block_effort_weighted=float(np.average(vb.days,weights=vb.hours)),share_effort_vessels_3plus_days=float(vb[vb.days>=3].hours.sum()/vb.hours.sum()),
                  share_effort_vessels_7plus_days=float(vb[vb.days>=7].hours.sum()/vb.hours.sum()),vessels_per_month=float(dvt.groupby('t').vessel.nunique().mean()),daily_vs_monthly_hours=float(dvt.hours.sum()/B_all[:,test_t].sum()))
        RESULTS['persistence']=PERS; print(json.dumps(PERS,indent=1))
        # ---- the fleet by year: did the vessels move, or did the coverage of the data change? (reachability ceiling)
        dv['year']=dv.date.dt.year; dv['beyond']=dv.blk_id.map(pd.Series(1-blocks.reachable.values,index=blocks.blk_id.values)).astype(int); dv['oce']=dv.blk_id.map(pd.Series((blocks.region=='oceanic').values,index=blocks.blk_id.values))
        do=dv[dv.oce]; fy=do.groupby('year').agg(hours=('hours','sum'),vessels=('vessel','nunique'),vessel_days=('hours','size')).reset_index()
        fy['hours_beyond_reach']=fy.year.map(do[do.beyond==1].groupby('year').hours.sum()).fillna(0.0); fy['vessels_beyond_reach']=fy.year.map(do[do.beyond==1].groupby('year').vessel.nunique()).fillna(0).astype(int)
        fy['hours_per_vessel_day']=fy.hours/fy.vessel_days; early=set(do[(do.beyond==1)&(do.year<=2022)].vessel.unique()); late=do[(do.year>=2023)&(do.beyond==0)]
        RESULTS['fleet_by_year']=dict(rows=fy.round(2).to_dict('records'),vessels_beyond_reach_to_2022=len(early),share_of_them_seen_within_reach_2023on=float(len(early&set(late.vessel.unique()))/max(len(early),1)),
                                      share_of_within_reach_effort_2023on_from_them=float(late[late.vessel.isin(early)].hours.sum()/max(late.hours.sum(),1e-9)))
        print(fy.round(1).to_string(index=False)); print({k_:v_ for k_,v_ in RESULTS['fleet_by_year'].items() if k_!='rows'})
        # ---- presence probability by block from the trailing 12 months (information available before t)
        bmall=dv.groupby(['blk_id','t']).agg(hours=('hours','sum'),days=('day','nunique'),dim=('dim','first')).reset_index(); bmall['presence']=bmall.days/bmall.dim
        def presence_vec(t):
            h_=bmall[(bmall.t<t)&(bmall.t>=t-12)]
            if not len(h_): return np.full(NB,PERS['presence_effort_weighted'])
            g_=h_.groupby('blk_id').apply(lambda d: np.average(d.presence,weights=d.hours)); q_=np.full(NB,float(np.average(h_.presence,weights=h_.hours))); q_[g_.index.values]=g_.values
            return np.clip(q_,0.05,1.0)
        # ---- dated backtest: each sortie gets a day; a search meets the vessels that fished in the block on that day
        by_t={t:d for t,d in dvt.groupby('t')}
        # blocks and years for which vessel-level data were downloaded: the monthly score is restricted to them for the comparison
        COVB=np.ones((NB,NT//12+1),bool)
        if os.path.exists(f'{AD}/daily_coverage.csv'):
            cv=pd.read_csv(f'{AD}/daily_coverage.csv').round({'blk_lon':4,'blk_lat':4}).merge(blocks[['blk_lon','blk_lat','blk_id']].round({'blk_lon':4,'blk_lat':4}),on=['blk_lon','blk_lat'])
            COVB[:]=False; COVB[cv.blk_id.values,(cv.year.values-Y0)]=True
        RESULTS['daily_coverage_share']=float(sum((B_all[:,t]*COVB[:,t//12]).sum() for t in test_t)/B_all[:,test_t].sum()); print('share of the test-period effort in blocks with vessel-level data:',round(RESULTS['daily_coverage_share'],3))
        def dated_score(routes,t):
            """routes: list of (base, platform, blocks) in planning order. Sorties of each aircraft group are spread evenly over the month."""
            d=by_t.get(t)
            if d is None or not routes: return dict(IA_dated=0.0,IA_vessel=0.0,vessels=0.0,vessels_total=0 if d is None else d.vessel.nunique(),encounters=0.0)
            dim=int(d.dim.iloc[0]); grp={}
            for i,(b,p,vis) in enumerate(routes): grp.setdefault((b,p),[]).append(vis)
            sb=[]; sd=[]
            for g_,lst in grp.items():
                for j,vis in enumerate(lst):
                    day=int(np.floor((j+0.5)*dim/len(lst)))+1; sb+=list(vis); sd+=[day]*len(vis)
            se=pd.DataFrame({'blk_id':sb,'day':sd}).groupby(['blk_id','day']).size().rename('k').reset_index()
            m_=d.merge(se,on=['blk_id','day'],how='inner')                                # vessel-days that coincide with a search of their block
            vbm=d.groupby(['vessel','blk_id']).hours.sum().rename('h').reset_index()        # monthly hours of each vessel in each block
            if not len(m_): return dict(IA_dated=0.0,IA_vessel=0.0,vessels=0.0,vessels_total=d.vessel.nunique(),encounters=0.0)
            kb=m_.groupby(['vessel','blk_id']).k.sum().reset_index(); kb['p']=1-(1-PMIN)**kb.k     # detected in that block at least once
            ia_b=float(kb.merge(vbm,on=['vessel','blk_id']).eval('p*h').sum())
            kv=m_.groupby('vessel').k.sum(); pv=1-(1-PMIN)**kv
            ia_v=float((pv*d.groupby('vessel').hours.sum().reindex(pv.index)).sum())
            return dict(IA_dated=ia_b,IA_vessel=ia_v,vessels=float(pv.sum()),vessels_total=int(d.vessel.nunique()),encounters=float(PMIN*m_.k.sum()))
        _c=cache_load('dated2')
        if _c is None:
            _c=[]; t0=time.time()
            for sc in ['S0','S5','S*']:
                for mt in (['risk','hist','oracle'] if FAST_MODE else ['risk','hist','uniform','oracle']):
                    for t in test_t:
                        res=evaluate(SCENARIOS[sc],mt,t,keep_routes=True); _c.append(dict(scenario=sc,method=mt,t=t,IA=float((B_all[:,t]*res['frac']*COVB[:,t//12]).sum()),IA_all=res['IA'],**dated_score(res['routes'],t)))
                print(f'  dated backtest {sc}: {(time.time()-t0)/60:.1f} min',flush=True)
            for sc in ['S0','S5']:                                                          # plans that use the block-specific presence probability
                for t in test_t:
                    _q=PRESENCE; PRESENCE=presence_vec(t)
                    try: res=evaluate(SCENARIOS[sc],'risk',t,keep_routes=True)
                    finally: PRESENCE=_q
                    _c.append(dict(scenario=sc,method='risk, presence-aware',t=t,IA=np.nan,**dated_score(res['routes'],t)))
            cache_save('dated2',_c)
        dd=pd.DataFrame(_c); dd.to_csv(f'{OUT}/dated_backtest_monthly.csv',index=False)
        tot_h=dvt.groupby('t').hours.sum().reindex(test_t).fillna(0).values
        T16=dd.groupby(['scenario','method']).agg(IA_model=('IA','mean'),IA_dated=('IA_dated','mean'),IA_vessel=('IA_vessel','mean'),vessels=('vessels','mean'),vessels_total=('vessels_total','mean'),encounters=('encounters','mean')).reset_index()
        T16['dated_over_model']=T16.IA_dated/T16.IA_model; T16['share_dated']=T16.IA_dated/tot_h.mean(); T16['share_vessel']=T16.IA_vessel/tot_h.mean(); T16['vessel_share']=T16.vessels/T16.vessels_total
        def dser(sc,mt,col): return dd[(dd.scenario==sc)&(dd.method==mt)].set_index('t')[col].reindex(test_t).values
        CMP=[]
        for lab,(s1,m1),(s2,m2) in [('S5 vs S0 (risk)',('S5','risk'),('S0','risk')),('S* vs S0 (risk)',('S*','risk'),('S0','risk')),('risk vs historical, S0',('S0','risk'),('S0','hist')),('risk vs historical, S5',('S5','risk'),('S5','hist')),
                                    ('presence-aware vs baseline plan, S0',('S0','risk, presence-aware'),('S0','risk')),('presence-aware vs baseline plan, S5',('S5','risk, presence-aware'),('S5','risk'))]+([] if FAST_MODE else [('risk vs uniform, S0',('S0','risk'),('S0','uniform'))]):
            for col in ['IA_dated','IA_vessel','vessels']:
                a=dser(s1,m1,col); b=dser(s2,m2,col); lo,hi=boot_ratio_ci(a,b); CMP.append(dict(comparison=lab,metric=col,gain=100*(a.mean()/b.mean()-1),lo=lo,hi=hi,wins=int((a>b).sum())))
        T16c=pd.DataFrame(CMP); print(T16.round(3).to_string(index=False)); print(T16c.round(1).to_string(index=False))
        T16.to_csv(f'{OUT}/T16_dated.csv',index=False); T16c.to_csv(f'{OUT}/T16_dated_comparisons.csv',index=False); RESULTS['T16']=T16.round(5).to_dict('records'); RESULTS['T16c']=T16c.round(4).to_dict('records')
        # ---- no-take detail
        if os.path.exists(f'{AD}/notake_daily_detail.csv'):
            ntd=pd.read_csv(f'{AD}/notake_daily_detail.csv',parse_dates=['date']); ntd['year']=ntd.date.dt.year
            NTD=dict(hours=float(ntd.hours.sum()),vessels=int(ntd.vessel.nunique()),vessel_days=int(ntd.groupby(['vessel','date']).ngroups),
                     by_site=ntd.groupby('mpa_name').agg(hours=('hours','sum'),vessels=('vessel','nunique'),days=('date','nunique')).round(1).reset_index().to_dict('records'),
                     by_flag=ntd.groupby('flag').agg(hours=('hours','sum'),vessels=('vessel','nunique')).sort_values('hours',ascending=False).round(1).reset_index().to_dict('records'),
                     by_gear=ntd.groupby('geartype').hours.sum().sort_values(ascending=False).round(1).to_dict(),by_year=ntd.groupby('year').agg(hours=('hours','sum'),vessels=('vessel','nunique')).round(1).reset_index().to_dict('records'),
                     top5_vessels_share=float(ntd.groupby('vessel').hours.sum().nlargest(5).sum()/ntd.hours.sum()),hours_per_vessel_day_median=float(ntd.groupby(['vessel','date']).hours.sum().median()),
                     share_hours_in_vessel_days_under_2h=float(ntd.groupby(['vessel','date']).hours.sum().pipe(lambda s_: s_[s_<2].sum()/s_.sum())))
            RESULTS['notake_detail']=NTD; print(json.dumps({k_:v_ for k_,v_ in NTD.items() if k_ not in ('by_site',)},indent=1,default=str)[:1500])
        # ---- SAR detections not matched to AIS
        if os.path.exists(f'{AD}/sar_monthly.parquet'):
            sm=pd.read_parquet(f'{AD}/sar_monthly.parquet'); sm['t']=(sm.year-Y0)*12+sm.month-1; sm=sm[(sm.t>=0)&(sm.t<NT)]
            SU=np.zeros((n,NT)); SMt=np.zeros((n,NT)); SU[sm.cell_id.values,sm.t.values]=sm.det_unmatched.values; SMt[sm.cell_id.values,sm.t.values]=sm.det_matched.values
            B_su=blk_agg(SU); B_sm=blk_agg(SMt); cov_b=(B_su+B_sm).sum(1)>0                  # blocks imaged at least once (any detection)
            SAR=dict(unmatched=float(SU.sum()),matched=float(SMt.sum()),blocks_with_detections=int(cov_b.sum()),
                     spearman_block_ais_vs_unmatched=float(stats.spearmanr(B_all[cov_b].sum(1),B_su[cov_b].sum(1)).correlation) if cov_b.sum()>10 else None,
                     share_unmatched_in_top_decile_ais_blocks=float(B_su[cov_b].sum(1)[np.argsort(-B_all[cov_b].sum(1))[:max(1,int(0.1*cov_b.sum()))]].sum()/max(B_su.sum(),1e-9)),
                     by_region={r_:dict(unmatched=float(SU[(cells.region==r_).values].sum()),matched=float(SMt[(cells.region==r_).values].sum())) for r_ in cells.region.unique()})
            _c=cache_load('sar_bt')
            if _c is None:
                _c={}
                for sc in ['S0','S5','S*']:
                    for mt in ['risk','hist']+([] if FAST_MODE else ['uniform']):
                        _c[(sc,mt)]=np.array([float((B_su[:,t]*evaluate(SCENARIOS[sc],mt,t)['frac']).sum()) for t in test_t])
                cache_save('sar_bt',_c)
            su_tot=B_su[:,test_t].sum(0).mean(); SAR['unmatched_per_test_month']=float(su_tot)
            SAR['intercepted_share']={f'{sc}|{mt}':float(v_.mean()/max(su_tot,1e-9)) for (sc,mt),v_ in _c.items()}
            RESULTS['sar']=SAR; print(json.dumps(SAR,indent=1,default=str))
except Exception:
    RESULTS['stage_N_error']=traceback.format_exc(); print('STAGE N FAILED (the rest of the notebook continues):'); print(RESULTS['stage_N_error'])

## O. Costo de aleatorizar el plan
Se perturban las recompensas con ruido lognormal y se mide cuánto puntaje se pierde y cuánto cambia el conjunto de bloques patrullados.

In [ ]:
import traceback
try:
    _c=cache_load('random_plans')
    if _c is None:
        _c=[]; rng=np.random.default_rng(SEED+9); TT=test_t[::4] if FAST_MODE else test_t
        for sc in ['S0','S5']:
            for sg in [0.0,0.25,0.5,1.0]:
                for t in TT:
                    base=evaluate(SCENARIOS[sc],'risk',t); draws=[]
                    for _ in range(1 if sg==0 else 2):
                        r_=evaluate(SCENARIOS[sc],'x',t,reward=B_rhat[:,t]*np.exp(sg*rng.standard_normal(NB))); draws.append(r_)
                    jac=np.nan
                    if sg>0:
                        A_=draws[0]['nvis']>0; B_=draws[1]['nvis']>0; jac=float((A_&B_).sum()/max((A_|B_).sum(),1))
                    _c.append(dict(scenario=sc,sigma=sg,t=t,IA=float(np.mean([d_['IA'] for d_ in draws])),IA_base=base['IA'],jaccard=jac))
        cache_save('random_plans',_c)
    rp=pd.DataFrame(_c); T17=rp.groupby(['scenario','sigma']).apply(lambda d: pd.Series(dict(IA=d.IA.mean(),loss_pct=100*(1-d.IA.mean()/d.IA_base.mean()),overlap=d.jaccard.mean()))).reset_index()
    print(T17.round(3).to_string(index=False)); T17.to_csv(f'{OUT}/T17_randomization.csv',index=False); RESULTS['T17']=T17.round(5).to_dict('records')
except Exception:
    RESULTS.setdefault('stage_errors',{})['O']=traceback.format_exc(); print('STAGE O FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['O'])

## P. Verificaciones de datos y horas de traslado

In [ ]:
import traceback
try:
    GEAR=['h_trawl','h_longline','h_squid','h_seine','h_other']
    # ---- effort recorded inside the sites, by gear: fishing or support traffic? (minor 10)
    ins=oc[oc.zone=='interior'].groupby('mpa_name').agg(hours=('all_hours','sum'),cell_months=('all_hours',lambda s_:int((s_>0).sum())),vessels_cell_month_mean=('n_vessels',lambda s_:float(s_[s_>0].mean()) if (s_>0).any() else 0.0),
                                                        **{g_:(g_,'sum') for g_ in GEAR}).reset_index()
    ins=ins[ins.hours>0].sort_values('hours',ascending=False); ins['top_gear']=ins[GEAR].idxmax(1).str[2:]; ins['top_gear_share']=ins[GEAR].max(1)/ins[GEAR].sum(1).clip(lower=1e-9)
    print(ins.round(2).to_string(index=False)); RESULTS['inside_by_gear']=ins.round(3).to_dict('records')
    # ---- the change of 2023 (minor 6): effort by year and gear, within and beyond reach; active cells; the Nazca-Desventuradas ring by distance band
    yg=oc.groupby('year')[['all_hours']+GEAR].sum(); yg['active_cell_months']=oc[oc.all_hours>0].groupby('year').size(); yg['hours_per_active_cell_month']=yg.all_hours/yg.active_cell_months
    yg['beyond_reach']=oc[oc.reachable==0].groupby('year').all_hours.sum(); yg['beyond_reach_squid']=oc[oc.reachable==0].groupby('year').h_squid.sum()
    nz=oc[(oc.zone=='ring')&oc.mpa_name.str.contains('Nazca')].copy(); nz['band']=pd.cut(nz.dist_bnd_km,[0,200,400,600,800],labels=['0-200','200-400','400-600','600-800'],include_lowest=True)
    nzb=nz.pivot_table(index='year',columns='band',values='all_hours',aggfunc='sum',observed=False).fillna(0)
    print(yg.round(0).to_string()); print(nzb.round(0).to_string())
    RESULTS['by_year_detail']=yg.round(1).reset_index().to_dict('records'); RESULTS['nazca_ring_by_band']=nzb.round(1).reset_index().to_dict('records')
    RESULTS['oceanic_total_cv']=float(oc[oc.region=='oceanic'].groupby('year').all_hours.sum().pipe(lambda s_: s_.std()/s_.mean()))
    # ---- ferry hours of a deployment: one round trip per aircraft and month between the home base and its station, for maintenance and crew rotation
    def ferry_hours(scen,home='TQM'):
        h=0.0
        for b,ac in scen.items():
            d_=float(haversine_km(BASES.loc[b,'lon'],BASES.loc[b,'lat'],BASES.loc[home,'lon'],BASES.loc[home,'lat']))
            for p,k in ac.items(): h+=k*2*d_/PL[p]['v']
        return h
    FER={}
    for sc in ['S1','S2','S3','S4','S5','S*']:
        fh=float(S[(S.scenario==sc)&(S.method=='risk')].flight_h.iloc[0]) if ((S.scenario==sc)&(S.method=='risk')).any() else float(RESULTS.get('star_full',{}).get('flight_h',float('nan')))
        FER[sc]=dict(ferry_h=ferry_hours(SCENARIOS[sc]),flight_h=fh)
        FER[sc]['ferry_pct_of_flight_h']=100*FER[sc]['ferry_h']/fh if fh==fh else float('nan')
    RESULTS['ferry']=FER; print(pd.DataFrame(FER).T.round(1).to_string())
except Exception:
    RESULTS.setdefault('stage_errors',{})['P']=traceback.format_exc(); print('STAGE P FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['P'])

## Resumen y descarga

In [ ]:
import traceback
try:
    RESULTS['config']=dict(CREW_H=CREW_H,PLATFORMS=PLATFORMS.reset_index().to_dict('records'),SORTIE_SEEDS=SORTIE_SEEDS,SORTIE_SWAP=SORTIE_SWAP,REOPT_PASSES=REOPT_PASSES,FAST_MODE=FAST_MODE,TEST_YEARS=TEST_YEARS,SEL_YEAR=SEL_YEAR,BLK_DEG=BLK_DEG,PMIN=PMIN,PRESENCE=PRESENCE,RESERVE=RESERVE,K_POOL=K_POOL,
                           SORTIES_PER_AC_MONTH=SORTIES_PER_AC_MONTH,COVER_BONUS=COVER_BONUS,W_NT=W_NT,n_blocks=int(NB),N_BOOT=N_BOOT,LP_LEVELS=LP_LEVELS,
                           versions=dict(numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,lightgbm=lgb.__version__))
    json.dump(RESULTS,open(f'{OUT}/results.json','w'),indent=1,default=str)
    L=[]; A=L.append
    A('='*78); A(f'SUMMARY NOTEBOOK 2 v2 ({"FAST" if FAST_MODE else "FULL"})  test={TEST_YEARS}'); A('='*78)
    A('--- A. boundary ---'); A(json.dumps(RESULTS['boundary'],indent=1))
    A('--- B. risk (mean over test years) ---'); A(T2.round(3).to_string(index=False))
    A('--- D. methods ---'); A(T3.round(4).to_string(index=False)); A(f'Friedman S0: {fr}')
    A('--- D. scenarios (risk) ---'); A(T4.round(4).to_string(index=False)); A(f'oracle S6 vs S5: {RESULTS["oracle_S6_vs_S5_pct"]:+.2f}%')
    A('--- E. LP upper bound ---'); A(T7.round(4).to_string(index=False))
    A('--- F. single-sortie benchmark ---'); A(T8.round(4).to_string(index=False))
    A('--- G. basing ---'); A(json.dumps(STAR,indent=1,default=str)); A(top.round(3).to_string(index=False))
    A('--- H. sensitivity ---'); A(T6.round(3).to_string(index=False))
    for k_ in ['jurisdiction','T12','T13','T13b','T14','T14_levels','T14_uniform','lp_ablation','intensive','T15','T15b','T15c','exact_op_list_growth','stage_L2_error','star_full','coverage_indicator','rolling_selection','bootstrap_blocks','persistence','T16','T16c','notake_detail','sar','T17','stage_N_error','daily_coverage_share','fleet_by_year','inside_by_gear','by_year_detail','nazca_ring_by_band','ferry','blocksize_n']:
        if k_ in RESULTS: A(f'--- {k_} ---'); A(pd.DataFrame(RESULTS[k_]).round(4).to_string(index=False) if isinstance(RESULTS[k_],list) and RESULTS[k_] and isinstance(RESULTS[k_][0],dict) else json.dumps(RESULTS[k_],indent=1,default=str))
    A(f'realized per test month: all {RESULTS["all_realized_per_month_test"]:.0f} h, no-take {NT_REALIZED_PM:.2f} h; unreachable share by year {RESULTS["unreachable_share_by_year"]}')
    if RESULTS.get('stage_errors') or RESULTS.get('stage_N_error') or RESULTS.get('stage_L2_error'):
        A('='*78); A('ATENCION: hubo etapas con error -> '+', '.join(list(RESULTS.get('stage_errors',{}).keys())+(['N'] if RESULTS.get('stage_N_error') else [])+(['L2'] if RESULTS.get('stage_L2_error') else [])))
        for k_,v_ in RESULTS.get('stage_errors',{}).items(): A(f'--- error stage {k_} ---'); A(v_)
    txt='\n'.join(L); print(txt); open(f'{OUT}/SUMMARY.txt','w').write(txt)
    try:
        import zipfile; z=OUT+'.zip'
        with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as zf:
            for f_ in sorted(os.listdir(OUT)):
                if not f_.startswith('cache_') and not f_.endswith('.npy'): zf.write(f'{OUT}/{f_}',f_)
            zf.write(f'{OUT}/basing_oos_IA.npy','basing_oos_IA.npy')
        from google.colab import files; files.download(z)
    except Exception as e: print('manual download:',OUT,'|',e)
except Exception:
    RESULTS.setdefault('stage_errors',{})['P']=traceback.format_exc(); print('STAGE P FAILED (recorded; the notebook continues):'); print(RESULTS['stage_errors']['P'])